In [2]:
import psutil
print(f"Available RAM: {psutil.virtual_memory().available / 1024**3:.1f} GB")

Available RAM: 1.1 GB


In [3]:

# ============================================================
# CELL 1: Imports & Configuration
# ============================================================

import sqlite3
import pandas as pd
import numpy as np
import warnings
import time
import math
from datetime import datetime

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('future.no_silent_downcasting', True)

DB_PATH = r'C:\Users\Sarthak\Documents\ML\fighter-beta\mma_fighters.db'
print("✔ Imports complete")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Sarthak\miniconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Sarthak\miniconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Sarthak\miniconda3\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "C:\Users\Sarthak\miniconda3\L

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Sarthak\miniconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Sarthak\miniconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Sarthak\miniconda3\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "C:\Users\Sarthak\miniconda3\L

AttributeError: _ARRAY_API not found

✔ Imports complete


In [4]:

# ============================================================
# CELL 2: Database Connection
# ============================================================

conn = sqlite3.connect(DB_PATH)

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f"✔ Connected. Tables: {len(tables)}")

fights_count = pd.read_sql("SELECT COUNT(*) as c FROM fights_v2", conn)['c'].iloc[0]
print(f"✔ Total fights in DB: {fights_count}")


✔ Connected. Tables: 10
✔ Total fights in DB: 11127


In [5]:


# ============================================================
# CELL 3: Helper Functions
# ============================================================

def parse_reach(r):
    if pd.isna(r) or r == '--': return None
    try: return float(r.replace('"', ''))
    except: return None

def parse_height(h):
    if pd.isna(h) or h == '--': return None
    try:
        parts = h.replace('"', '').split("'")
        return int(parts[0]) * 12 + int(parts[1])
    except: return None

def parse_age(dob, fight_date):
    if pd.isna(dob) or pd.isna(fight_date): return None
    try:
        birth = datetime.strptime(dob, "%b %d, %Y")
        fight = datetime.strptime(fight_date, "%Y-%m-%d")
        return (fight - birth).days / 365.25
    except: return None

def parse_fight_duration(ending_round, ending_time):
    try:
        mins, secs = ending_time.split(':')
        final_round_minutes = int(mins) + int(secs) / 60
        return ((ending_round - 1) * 5) + final_round_minutes
    except: return 15.0

# Time decay: 1.5 year half-life (λ ≈ 0.00127)
LAM = math.log(2) / (1.5 * 365)

def time_decay_weights(dates, as_of_date, lam=LAM):
    as_of = datetime.strptime(as_of_date, "%Y-%m-%d")
    weights = []
    for d in dates:
        fight_dt = datetime.strptime(d, "%Y-%m-%d")
        days_ago = (as_of - fight_dt).days
        w = np.exp(-lam * max(days_ago, 0))
        weights.append(w)
    weights = np.array(weights)
    return weights / weights.sum() if weights.sum() > 0 else weights

def kish_effective_n(weights):
    """Kish effective sample size — more honest than raw count with decayed weights"""
    if weights.sum() == 0: return 0
    return (weights.sum() ** 2) / (weights ** 2).sum()
    
def normalize_weight_class(wc):
    if pd.isna(wc): return None
    wc = wc.strip()
    
    # Drop women's divisions
    if "Women's" in wc: return None
    
    # Map to base weight classes
    base_classes = [
        'Heavyweight', 'Light Heavyweight', 'Middleweight',
        'Welterweight', 'Lightweight', 'Featherweight',
        'Bantamweight', 'Flyweight', 'Catch Weight'
    ]
    for base in base_classes:
        if base in wc:
            return f'{base} Bout'
    
    return None

print(f"✔ Lambda (1.5yr half-life): {LAM:.6f}")
print("✔ Helper functions ready")

✔ Lambda (1.5yr half-life): 0.001266
✔ Helper functions ready


In [6]:
# ============================================================
# CELL 4: Pre-Cache Fight Data (with Poisson-Gamma & Beta-Binomial smoothing)
# ============================================================

print("Fetching all fight stats (no date filter)...")
start = time.time()

# ── Raw stats ──────────────────────────────────────────────
all_fight_stats_raw = pd.read_sql("""
    SELECT 
        f.fight_id,
        f.event_date,
        f.weight_class,
        f.ending_round,
        f.ending_time,
        f.method,
        ff.fighter_id,
        SUM(frs.sig_str_landed)    as sig_str_landed,
        SUM(frs.sig_str_attempted) as sig_str_attempted,
        SUM(frs.td_landed)         as td_landed,
        SUM(frs.td_attempted)      as td_attempted,
        SUM(frs.sub_attempts)      as sub_attempts,
        SUM(frs.ctrl_seconds)      as ctrl_seconds,
        SUM(frs.knockdowns)        as knockdowns
    FROM fight_round_stats_v2 frs
    JOIN fight_rounds_v2 fr      ON SUBSTR(frs.round_id, 1, INSTR(frs.round_id, ':') - 1) = fr.fight_id
    JOIN fights_v2 f             ON fr.fight_id   = f.fight_id
    JOIN fight_fighters_v2 ff    ON f.fight_id    = ff.fight_id 
                                AND frs.fighter_id = ff.fighter_id
    GROUP BY f.fight_id, ff.fighter_id
""", conn)

# Actual fight duration
all_fight_stats_raw['actual_minutes'] = all_fight_stats_raw.apply(
    lambda r: parse_fight_duration(r['ending_round'], r['ending_time']), axis=1
)
all_fight_stats_raw = all_fight_stats_raw.replace([np.inf, -np.inf], np.nan).fillna(0)

# ── Step 1: Compute weight-class rate priors ─────────────────
print("Computing weight-class priors...")

all_fight_stats_raw['wc_norm'] = all_fight_stats_raw['weight_class'].apply(normalize_weight_class)
valid = all_fight_stats_raw[all_fight_stats_raw['wc_norm'].notna()].copy()

pg_stats = {
    'sig_str_landed': 0.7,
    'td_landed':      7.0,
    'sub_attempts':   12.0,
    'knockdowns':     20.0,
    'ctrl_seconds':   2.0,
}

wc_rates     = {}
global_rates = {}

for stat, tau in pg_stats.items():
    global_total       = valid[stat].sum()
    global_mins        = valid['actual_minutes'].sum()
    global_rates[stat] = global_total / global_mins if global_mins > 0 else 0

    for wc, grp in valid.groupby('wc_norm'):
        if wc not in wc_rates:
            wc_rates[wc] = {}
        total = grp[stat].sum()
        mins  = grp['actual_minutes'].sum()
        wc_rates[wc][stat] = total / mins if mins > 0 else global_rates[stat]

bb_stats = {
    'str_acc': {'num': 'sig_str_landed', 'den': 'sig_str_attempted', 'tau': 0.7},
    'td_acc':  {'num': 'td_landed',      'den': 'td_attempted',      'tau': 7.0},
}

wc_bb_priors     = {}
global_bb_priors = {}

for stat, cfg in bb_stats.items():
    global_num = valid[cfg['num']].sum()
    global_den = valid[cfg['den']].sum()
    global_bb_priors[stat] = global_num / global_den if global_den > 0 else 0.5

    for wc, grp in valid.groupby('wc_norm'):
        if wc not in wc_bb_priors:
            wc_bb_priors[wc] = {}
        num = grp[cfg['num']].sum()
        den = grp[cfg['den']].sum()
        wc_bb_priors[wc][stat] = num / den if den > 0 else global_bb_priors[stat]

print(f"✓ Weight class priors computed for {len(wc_rates)} classes")

# ── Step 2: Apply smoothing ────────────────────────────────
print("Applying Poisson-Gamma and Beta-Binomial smoothing...")

all_fight_stats = all_fight_stats_raw.copy()

def get_wc_rate(wc, stat):
    wc_norm = normalize_weight_class(wc) if wc else None
    if wc_norm and wc_norm in wc_rates and stat in wc_rates[wc_norm]:
        return wc_rates[wc_norm][stat]
    return global_rates.get(stat, 0)

def get_wc_bb(wc, stat):
    wc_norm = normalize_weight_class(wc) if wc else None
    if wc_norm and wc_norm in wc_bb_priors and stat in wc_bb_priors[wc_norm]:
        return wc_bb_priors[wc_norm][stat]
    return global_bb_priors.get(stat, 0.5)

for stat, tau in pg_stats.items():
    smoothed = []
    for _, row in all_fight_stats.iterrows():
        t         = max(row['actual_minutes'], 0.1)
        wc_rate   = get_wc_rate(row['weight_class'], stat)
        x         = row[stat]
        post_rate = (wc_rate * tau + x) / (tau + t)
        smoothed.append(t * post_rate)
    all_fight_stats[f'{stat}_smooth'] = smoothed

for stat, cfg in bb_stats.items():
    smoothed = []
    for _, row in all_fight_stats.iterrows():
        wc_rate = get_wc_bb(row['weight_class'], stat)
        tau     = cfg['tau']
        x       = row[cfg['num']]
        n       = row[cfg['den']]
        if n == 0:
            smoothed.append(wc_rate)
        else:
            smoothed.append((wc_rate * tau + x) / (tau + n))
    all_fight_stats[f'{stat}_smooth'] = smoothed

# ── Step 3: Compute per-minute rates from smoothed values ──
t = all_fight_stats['actual_minutes'].clip(lower=0.1)

all_fight_stats['slpm']              = all_fight_stats['sig_str_landed_smooth'] / t
all_fight_stats['str_acc']           = all_fight_stats['str_acc_smooth']
all_fight_stats['td_acc']            = all_fight_stats['td_acc_smooth']
all_fight_stats['td_avg']            = all_fight_stats['td_landed_smooth']    / (t / 15)
all_fight_stats['sub_avg']           = all_fight_stats['sub_attempts_smooth'] / (t / 15)
all_fight_stats['ctrl_time_per_min'] = all_fight_stats['ctrl_seconds_smooth'] / 60 / t
all_fight_stats['kd_per_min']        = all_fight_stats['knockdowns_smooth']   / t

all_fight_stats = all_fight_stats.replace([np.inf, -np.inf], np.nan).fillna(0)
all_fight_stats = all_fight_stats.sort_values('event_date').reset_index(drop=True)

# ── Step 4: Rebuild dicts ──────────────────────────────────
fighter_fights_dict = {
    fid: grp.sort_values('event_date').reset_index(drop=True)
    for fid, grp in all_fight_stats.groupby('fighter_id')
}

all_opponents = pd.read_sql("""
    SELECT ff1.fight_id, ff1.fighter_id, ff2.fighter_id as opponent_id
    FROM fight_fighters_v2 ff1
    JOIN fight_fighters_v2 ff2 
        ON ff1.fight_id = ff2.fight_id 
       AND ff1.fighter_id != ff2.fighter_id
""", conn)

opponents_dict = {
    (row['fight_id'], row['fighter_id']): row['opponent_id']
    for _, row in all_opponents.iterrows()
}

print(f"Done in {time.time()-start:.1f}s")
print(f"✓ Fighters cached: {len(fighter_fights_dict)}")
print(f"✓ Fight records:   {len(all_fight_stats)}")
print(f"✓ Opponent lookups:{len(opponents_dict)}")
print(f"✓ Smoothing: Poisson-Gamma for counts, Beta-Binomial for accuracies")

Fetching all fight stats (no date filter)...
Computing weight-class priors...
✓ Weight class priors computed for 8 classes
Applying Poisson-Gamma and Beta-Binomial smoothing...
Done in 10.8s
✓ Fighters cached: 3760
✓ Fight records:   21568
✓ Opponent lookups:22254
✓ Smoothing: Poisson-Gamma for counts, Beta-Binomial for accuracies


In [7]:
# ============================================================
# CELL 4b: Pre-Cache Strike Breakdown + Derived Ratio Features
# ============================================================
# NEW: Added exchange_ratio, targeting_share, strike_share
# These are computed per-fight using both fighter + opponent data,
# then stored for later dec_avg + adjperf treatment in Cell 6b.

print("Fetching strike breakdown data...")
start = time.time()

strike_breakdown = pd.read_sql("""
    SELECT
        f.fight_id,
        f.event_date,
        f.weight_class,
        f.ending_round,
        f.ending_time,
        ff.fighter_id,
        SUM(ss.head_landed)         as head_landed,
        SUM(ss.head_attempted)      as head_attempted,
        SUM(ss.body_landed)         as body_landed,
        SUM(ss.body_attempted)      as body_attempted,
        SUM(ss.leg_landed)          as leg_landed,
        SUM(ss.leg_attempted)       as leg_attempted,
        SUM(ss.distance_landed)     as distance_landed,
        SUM(ss.distance_attempted)  as distance_attempted,
        SUM(ss.clinch_landed)       as clinch_landed,
        SUM(ss.clinch_attempted)    as clinch_attempted,
        SUM(ss.ground_landed)       as ground_landed,
        SUM(ss.ground_attempted)    as ground_attempted
    FROM fight_round_sig_strikes_v2 ss
    JOIN fight_rounds_v2 fr      ON ss.round_id   = fr.round_id
    JOIN fights_v2 f             ON fr.fight_id   = f.fight_id
    JOIN fight_fighters_v2 ff    ON f.fight_id    = ff.fight_id
                                AND ss.fighter_id  = ff.fighter_id
    GROUP BY f.fight_id, ff.fighter_id
""", conn)

# Add fight duration
strike_breakdown['actual_minutes'] = strike_breakdown.apply(
    lambda r: parse_fight_duration(r['ending_round'], r['ending_time']), axis=1
)

# Per-minute rates and accuracy
for pos in ['head', 'body', 'leg', 'distance', 'clinch', 'ground']:
    strike_breakdown[f'{pos}_lpm'] = (
        strike_breakdown[f'{pos}_landed'] / strike_breakdown['actual_minutes']
    )
    strike_breakdown[f'{pos}_acc'] = (
        strike_breakdown[f'{pos}_landed'] / strike_breakdown[f'{pos}_attempted']
    )

# Total sig strikes landed per fight (for derived ratios)
strike_breakdown['total_sig_landed'] = (
    strike_breakdown['head_landed'] +
    strike_breakdown['body_landed'] +
    strike_breakdown['leg_landed']
)

strike_breakdown = strike_breakdown.replace([np.inf, -np.inf], np.nan).fillna(0)
strike_breakdown = strike_breakdown.sort_values('event_date').reset_index(drop=True)

# ── Compute derived ratio features per fight ─────────────────
# These need opponent data from the same fight, so we join via opponents_dict

derived_rows = []
for fight_id in strike_breakdown['fight_id'].unique():
    fight_data = strike_breakdown[strike_breakdown['fight_id'] == fight_id]
    if len(fight_data) != 2:
        continue

    for _, row in fight_data.iterrows():
        fid = row['fighter_id']
        opp_id = opponents_dict.get((fight_id, fid))
        if not opp_id:
            continue

        opp_row = fight_data[fight_data['fighter_id'] == opp_id]
        if len(opp_row) == 0:
            continue
        opp_row = opp_row.iloc[0]

        # Exchange ratio: fighter_sig_landed / opponent_sig_landed
        opp_landed = opp_row['total_sig_landed']
        exchange_ratio = (
            row['total_sig_landed'] / opp_landed
            if opp_landed > 0 else 1.0
        )

        # Targeting share: head_landed / total_sig_str_landed
        targeting_share = (
            row['head_landed'] / row['total_sig_landed']
            if row['total_sig_landed'] > 0 else 0.0
        )

        # Strike share: own_sig_str / (own + opponent sig_str)
        total_both = row['total_sig_landed'] + opp_row['total_sig_landed']
        strike_share = (
            row['total_sig_landed'] / total_both
            if total_both > 0 else 0.5
        )

        derived_rows.append({
            'fight_id': fight_id,
            'fighter_id': fid,
            'exchange_ratio': float(np.clip(exchange_ratio, 0, 10)),
            'targeting_share': float(targeting_share),
            'strike_share': float(strike_share),
        })

derived_df = pd.DataFrame(derived_rows)

# Merge back into strike_breakdown
strike_breakdown = strike_breakdown.merge(
    derived_df, on=['fight_id', 'fighter_id'], how='left'
)
strike_breakdown[['exchange_ratio', 'targeting_share', 'strike_share']] = (
    strike_breakdown[['exchange_ratio', 'targeting_share', 'strike_share']].fillna(0)
)

# Dict keyed by fighter_id for O(1) lookups
strike_breakdown_dict = {
    fid: grp.sort_values('event_date').reset_index(drop=True)
    for fid, grp in strike_breakdown.groupby('fighter_id')
}

print(f"Done in {time.time()-start:.1f}s")
print(f"✓ Strike breakdown records: {len(strike_breakdown)}")
print(f"✓ Fighters with strike data: {len(strike_breakdown_dict)}")
print(f"✓ NEW derived features: exchange_ratio, targeting_share, strike_share")
print(f"  exchange_ratio  mean={strike_breakdown['exchange_ratio'].mean():.3f}")
print(f"  targeting_share mean={strike_breakdown['targeting_share'].mean():.3f}")
print(f"  strike_share    mean={strike_breakdown['strike_share'].mean():.3f}")

Fetching strike breakdown data...
Done in 34.7s
✓ Strike breakdown records: 21568
✓ Fighters with strike data: 3760
✓ NEW derived features: exchange_ratio, targeting_share, strike_share
  exchange_ratio  mean=1.520
  targeting_share mean=0.597
  strike_share    mean=0.499


In [8]:
# ============================================================
# CELL 4c: Pre-Cache Strike Defense (what opponents land on each fighter)
# ============================================================

print("Building strike defense lookup...")
start = time.time()

# For each fight, get opponent's strike breakdown against this fighter
strike_defense = strike_breakdown.copy()

# Map opponent's stats to the fighter they were landed on
opp_strike_rows = []
for _, row in strike_defense.iterrows():
    opp_id = opponents_dict.get((row['fight_id'], row['fighter_id']))
    if opp_id:
        opp_strike_rows.append({
            'fight_id':         row['fight_id'],
            'event_date':       row['event_date'],
            'fighter_id':       opp_id,          # the fighter who RECEIVED these strikes
            'head_allowed':     row['head_lpm'],
            'body_allowed':     row['body_lpm'],
            'leg_allowed':      row['leg_lpm'],
            'distance_allowed': row['distance_lpm'],
            'clinch_allowed':   row['clinch_lpm'],
            'ground_allowed':   row['ground_lpm'],
        })

strike_defense_df = pd.DataFrame(opp_strike_rows).sort_values('event_date').reset_index(drop=True)

# Dict keyed by fighter_id
strike_defense_dict = {
    fid: grp.sort_values('event_date').reset_index(drop=True)
    for fid, grp in strike_defense_df.groupby('fighter_id')
}

print(f"Done in {time.time()-start:.1f}s")
print(f"✔ Defense records: {len(strike_defense_df)}")
print(f"✔ Fighters with defense data: {len(strike_defense_dict)}")

Building strike defense lookup...
Done in 2.0s
✔ Defense records: 21568
✔ Fighters with defense data: 3761


In [9]:
# ============================================================
# CELL 4d: Pre-Cache Fighter Results
# ============================================================

print("Building fighter results cache...")

results_raw = pd.read_sql("""
    SELECT 
        ff.fighter_id,
        ff.fight_id,
        ff.result,
        f.method
    FROM fight_fighters_v2 ff
    JOIN fights_v2 f ON ff.fight_id = f.fight_id
""", conn)

# Map (fighter_id, fight_id) → result
# Tag KO wins separately for ko_rate
fighter_results_dict = {}
for _, row in results_raw.iterrows():
    result = row['result']
    if result == 'win' and row['method'] == 'KO/TKO':
        tagged = 'ko_win'
    else:
        tagged = result
    fighter_results_dict[(row['fighter_id'], row['fight_id'])] = tagged

print(f"✔ Results cached: {len(fighter_results_dict)}")

Building fighter results cache...
✔ Results cached: 22254


In [10]:
# ============================================================
# CELL 4e: Pre-Cache Round 1 Stats (with R1 Strike Breakdown)
# ============================================================
# NEW: Also fetches R1 head/body/leg/clinch from
# fight_round_sig_strikes_v2 and joins into r1_stats.
# These give R1-specific strike targeting features.

print("Fetching Round 1 stats...")
start = time.time()

r1_stats = pd.read_sql("""
    SELECT
        f.fight_id,
        f.event_date,
        f.weight_class,
        f.ending_round,
        f.ending_time,
        ff.fighter_id,
        frs.sig_str_landed    as r1_sig_str_landed,
        frs.sig_str_attempted as r1_sig_str_attempted,
        frs.ctrl_seconds      as r1_ctrl_seconds,
        frs.td_landed         as r1_td_landed,
        frs.td_attempted      as r1_td_attempted,
        frs.knockdowns        as r1_knockdowns,
        frs.reversals         as r1_reversals,
        frs.sub_attempts      as r1_sub_attempts
    FROM fight_round_stats_v2 frs
    JOIN fights_v2 f          ON SUBSTR(frs.round_id, 1, INSTR(frs.round_id, ':') - 1) = f.fight_id
    JOIN fight_fighters_v2 ff ON f.fight_id    = ff.fight_id
                             AND frs.fighter_id = ff.fighter_id
    WHERE CAST(SUBSTR(frs.round_id, INSTR(frs.round_id, ':') + 1) AS INTEGER) = 1
""", conn)

# ── NEW: Fetch R1 sig strike breakdown (head/body/leg/clinch) ──
r1_sig_strikes = pd.read_sql("""
    SELECT
        f.fight_id,
        ss.fighter_id,
        ss.head_landed    as r1_head_landed,
        ss.head_attempted as r1_head_attempted,
        ss.body_landed    as r1_body_landed,
        ss.body_attempted as r1_body_attempted,
        ss.leg_landed     as r1_leg_landed,
        ss.leg_attempted  as r1_leg_attempted,
        ss.clinch_landed  as r1_clinch_landed,
        ss.clinch_attempted as r1_clinch_attempted,
        ss.distance_landed  as r1_distance_landed,
        ss.ground_landed    as r1_ground_landed
    FROM fight_round_sig_strikes_v2 ss
    JOIN fight_rounds_v2 fr ON ss.round_id = fr.round_id
    JOIN fights_v2 f        ON fr.fight_id = f.fight_id
    WHERE CAST(SUBSTR(ss.round_id, INSTR(ss.round_id, ':') + 1) AS INTEGER) = 1
""", conn)

# Join R1 sig strikes into r1_stats
r1_stats = r1_stats.merge(
    r1_sig_strikes, on=['fight_id', 'fighter_id'], how='left'
)
r1_sig_cols = [c for c in r1_sig_strikes.columns if c.startswith('r1_')]
r1_stats[r1_sig_cols] = r1_stats[r1_sig_cols].fillna(0)

# R1 exposure capped at 5 minutes
r1_stats['r1_minutes'] = r1_stats.apply(
    lambda r: min(parse_fight_duration(r['ending_round'], r['ending_time']), 5.0), axis=1
)
r1_stats['wc_norm'] = r1_stats['weight_class'].apply(normalize_weight_class)

r1_stats = r1_stats.replace([np.inf, -np.inf], np.nan).fillna(0)

# ── R1 Poisson-Gamma tau values (from author's blog) ──────
r1_pg_stats = {
    'r1_sig_str_landed': 0.7,
    'r1_td_landed':      9.0,
    'r1_sub_attempts':   15.0,
    'r1_knockdowns':     12.0,
    'r1_reversals':      60.0,
    # NEW: R1 strike breakdown counts
    'r1_head_landed':    1.0,
    'r1_body_landed':    3.0,
    'r1_leg_landed':     2.0,
    'r1_clinch_landed':  5.0,
}

# ── R1 weight-class rate priors ───────────────────────────
r1_valid = r1_stats[r1_stats['wc_norm'].notna()].copy()

r1_wc_rates     = {}
r1_global_rates = {}

for stat, tau in r1_pg_stats.items():
    total = r1_valid[stat].sum()
    mins  = r1_valid['r1_minutes'].sum()
    r1_global_rates[stat] = total / mins if mins > 0 else 0

    for wc, grp in r1_valid.groupby('wc_norm'):
        if wc not in r1_wc_rates:
            r1_wc_rates[wc] = {}
        wc_total = grp[stat].sum()
        wc_mins  = grp['r1_minutes'].sum()
        r1_wc_rates[wc][stat] = wc_total / wc_mins if wc_mins > 0 else r1_global_rates[stat]

# ── R1 Beta-Binomial for td_acc and ctrl ──────────────────
r1_bb_stats = {
    'r1_td_acc':  {'num': 'r1_td_landed', 'den': 'r1_td_attempted', 'tau': 9.0},
    'r1_ctrl':    {'num': 'r1_ctrl_seconds', 'den': 'r1_minutes', 'tau': 1.0},
}

r1_wc_bb     = {}
r1_global_bb = {}

for stat, cfg in r1_bb_stats.items():
    num_total = r1_valid[cfg['num']].sum()
    den_total = r1_valid[cfg['den']].sum()
    if stat == 'r1_ctrl':
        den_total = (r1_valid['r1_minutes'] * 60).sum()
    r1_global_bb[stat] = num_total / den_total if den_total > 0 else 0.5

    for wc, grp in r1_valid.groupby('wc_norm'):
        if wc not in r1_wc_bb:
            r1_wc_bb[wc] = {}
        num = grp[cfg['num']].sum()
        den = grp[cfg['den']].sum()
        if stat == 'r1_ctrl':
            den = (grp['r1_minutes'] * 60).sum()
        r1_wc_bb[wc][stat] = num / den if den > 0 else r1_global_bb[stat]

# ── Apply Poisson-Gamma smoothing to R1 counts ───────────
def get_r1_wc_rate(wc, stat):
    wc_n = normalize_weight_class(wc) if wc else None
    if wc_n and wc_n in r1_wc_rates and stat in r1_wc_rates[wc_n]:
        return r1_wc_rates[wc_n][stat]
    return r1_global_rates.get(stat, 0)

def get_r1_wc_bb(wc, stat):
    wc_n = normalize_weight_class(wc) if wc else None
    if wc_n and wc_n in r1_wc_bb and stat in r1_wc_bb[wc_n]:
        return r1_wc_bb[wc_n][stat]
    return r1_global_bb.get(stat, 0.5)

for stat, tau in r1_pg_stats.items():
    smoothed = []
    for _, row in r1_stats.iterrows():
        t       = max(row['r1_minutes'], 0.1)
        wc_rate = get_r1_wc_rate(row['weight_class'], stat)
        x       = row[stat]
        post    = (wc_rate * tau + x) / (tau + t)
        smoothed.append(t * post)
    r1_stats[f'{stat}_smooth'] = smoothed

# ── Apply Beta-Binomial to R1 td_acc and ctrl ────────────
for stat, cfg in r1_bb_stats.items():
    smoothed = []
    for _, row in r1_stats.iterrows():
        wc_rate = get_r1_wc_bb(row['weight_class'], stat)
        tau     = cfg['tau']
        x       = row[cfg['num']]
        if stat == 'r1_ctrl':
            n = row['r1_minutes'] * 60
        else:
            n = row[cfg['den']]
        if n == 0:
            smoothed.append(wc_rate)
        else:
            smoothed.append((wc_rate * tau + x) / (tau + n))
    r1_stats[f'{stat}_smooth'] = smoothed

# ── Compute per-minute rates from smoothed values ────────
t = r1_stats['r1_minutes'].clip(lower=0.1)

r1_stats['r1_slpm']           = r1_stats['r1_sig_str_landed_smooth'] / t
r1_stats['r1_td_per_min']     = r1_stats['r1_td_landed_smooth'] / t
r1_stats['r1_td_acc']         = r1_stats['r1_td_acc_smooth']
r1_stats['r1_ctrl_per_min']   = r1_stats['r1_ctrl_smooth']  # already a share
r1_stats['r1_kd_per_min']     = r1_stats['r1_knockdowns_smooth'] / t
r1_stats['r1_rev_per_min']    = r1_stats['r1_reversals_smooth'] / t
r1_stats['r1_sub_per_min']    = r1_stats['r1_sub_attempts_smooth'] / t
r1_stats['r1_td_att_per_min'] = r1_stats['r1_td_attempted'] / t

# NEW: R1 strike breakdown per-minute rates (smoothed)
r1_stats['r1_head_lpm']       = r1_stats['r1_head_landed_smooth'] / t
r1_stats['r1_body_lpm']       = r1_stats['r1_body_landed_smooth'] / t
r1_stats['r1_leg_lpm']        = r1_stats['r1_leg_landed_smooth'] / t
r1_stats['r1_clinch_lpm']     = r1_stats['r1_clinch_landed_smooth'] / t

r1_stats = r1_stats.replace([np.inf, -np.inf], np.nan).fillna(0)
r1_stats = r1_stats.sort_values('event_date').reset_index(drop=True)

r1_stats_dict = {
    fid: grp.sort_values('event_date').reset_index(drop=True)
    for fid, grp in r1_stats.groupby('fighter_id')
}

print(f"Done in {time.time()-start:.1f}s")
print(f"✔ R1 records: {len(r1_stats)}")
print(f"✔ Fighters with R1 data: {len(r1_stats_dict)}")
print(f"✔ R1 Poisson-Gamma smoothed: {list(r1_pg_stats.keys())}")
print(f"✔ R1 Beta-Binomial smoothed: {list(r1_bb_stats.keys())}")
print(f"✔ NEW R1 strike rates: r1_head_lpm, r1_body_lpm, r1_leg_lpm, r1_clinch_lpm")

Fetching Round 1 stats...
Done in 14.7s
✔ R1 records: 21568
✔ Fighters with R1 data: 3760
✔ R1 Poisson-Gamma smoothed: ['r1_sig_str_landed', 'r1_td_landed', 'r1_sub_attempts', 'r1_knockdowns', 'r1_reversals', 'r1_head_landed', 'r1_body_landed', 'r1_leg_landed', 'r1_clinch_landed']
✔ R1 Beta-Binomial smoothed: ['r1_td_acc', 'r1_ctrl']
✔ NEW R1 strike rates: r1_head_lpm, r1_body_lpm, r1_leg_lpm, r1_clinch_lpm


In [11]:
# ============================================================
# CELL 4f: Weight Class Priors
# ============================================================

print("Building weight class priors...")

STATS_FOR_PRIORS = ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg',
                    'ctrl_time_per_min', 'kd_per_min']

# Add normalized weight class to all_fight_stats
all_fight_stats['weight_class_norm'] = all_fight_stats['weight_class'].apply(normalize_weight_class)

# Filter to valid weight classes only
valid_stats = all_fight_stats[all_fight_stats['weight_class_norm'].notna()].copy()

# Compute per-weight-class median and MAD for each stat
wc_priors = {}
global_priors = {}

for stat in STATS_FOR_PRIORS:
    # Global prior as fallback
    global_median = float(np.median(valid_stats[stat].values))
    global_mad    = float(np.median(np.abs(valid_stats[stat].values - global_median)))
    global_priors[stat] = {'mean': global_median, 'mad': max(global_mad, 0.01)}

    # Per weight class
    for wc, grp in valid_stats.groupby('weight_class_norm'):
        if wc not in wc_priors:
            wc_priors[wc] = {}
        vals   = grp[stat].values
        median = float(np.median(vals))
        mad    = float(np.median(np.abs(vals - median)))
        wc_priors[wc][stat] = {'mean': median, 'mad': max(mad, 0.01)}

print(f"✔ Weight classes with priors: {len(wc_priors)}")
print(f"✔ Global fallback priors: {len(global_priors)} stats")
print("\nWeight classes:")
for wc in sorted(wc_priors.keys()):
    print(f"  {wc}")

Building weight class priors...
✔ Weight classes with priors: 8
✔ Global fallback priors: 7 stats

Weight classes:
  Bantamweight Bout
  Catch Weight Bout
  Featherweight Bout
  Flyweight Bout
  Heavyweight Bout
  Lightweight Bout
  Middleweight Bout
  Welterweight Bout


In [12]:
# ============================================================
# CELL 4g: Pre-Cache Per-Fight AdjPerf Z-Scores
# ============================================================
# NEW: After computing each fight's AdjPerf snapshot, store it
# so we can decay-average historical z-scores in Cell 6.
#
# WARNING: This cell is slow (~10-15 min) since it computes
# AdjPerf for every fighter across every fight historically.
# Run once and reuse fighter_adjperf_history dict.
#
# This resolves the key finding from the author's videos:
#   His naming: sig_str_land_ratio_dec_adjperf_dec_avg_diff
#   Means:      AdjPerf z-scores are ALSO decayed averaged
#               across recent fights, not just used as snapshots.
#
# Pipeline comparison:
#   Yours (before): dec_avg → AdjPerf snapshot → diff
#   His (confirmed): dec_avg → AdjPerf snapshot → decay avg → diff
WINDOW = 5
K_MEAN = 4.0
K_MAD  = 4.0

def bayesian_smooth(observed, n_eff, population_mean, k):
    w = n_eff / (n_eff + k)
    return w * observed + (1 - w) * population_mean

print("Pre-caching per-fight AdjPerf z-scores...")
start = time.time()
STATS = ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg',
         'ctrl_time_per_min', 'kd_per_min']
# Dict to store: {fighter_id: DataFrame with columns
#   [fight_id, event_date, {stat}_adjperf_snapshot]}
fighter_adjperf_history = {}

# We need to compute AdjPerf for every fight in all_fight_stats
# using only prior fight history — strict no-leakage
all_fights_sorted = all_fight_stats.sort_values('event_date').reset_index(drop=True)

pop_means = {s: float(np.median(all_fight_stats[s].values)) for s in STATS}
pop_mads  = {s: float(np.median(np.abs(
    all_fight_stats[s].values - pop_means[s]
))) for s in STATS}

adjperf_rows = []

for _, fight_row in all_fights_sorted.iterrows():
    fid   = fight_row['fighter_id']
    fdate = fight_row['event_date']
    fid_fight = fight_row['fight_id']

    # Get opponent
    opp_id = opponents_dict.get((fid_fight, fid))
    if not opp_id or opp_id not in fighter_fights_dict:
        continue

    fighter_hist = fighter_fights_dict[fid]
    opp_hist     = fighter_fights_dict[opp_id]

    fighter_prev = fighter_hist[fighter_hist['event_date'] < fdate].tail(WINDOW)
    opp_prev     = opp_hist[opp_hist['event_date'] < fdate]

    if len(fighter_prev) == 0:
        continue

    row = {'fighter_id': fid, 'fight_id': fid_fight, 'event_date': fdate}

    f_weights = time_decay_weights(fighter_prev['event_date'].tolist(), fdate)
    f_n_eff   = kish_effective_n(f_weights)

    for stat in STATS:
        # Layer 1: Fighter decayed avg
        dec_avg  = np.average(fighter_prev[stat].values, weights=f_weights)
        smoothed = bayesian_smooth(dec_avg, f_n_eff, pop_means[stat], K_MEAN)

        # Layer 2/3: Opponent allowed baseline → AdjPerf snapshot
        opp_allowed, opp_dates = [], []
        for _, opp_fight in opp_prev.iterrows():
            opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opp_id))
            if opp_opp_id and opp_opp_id in fighter_fights_dict:
                opp_opp_hist  = fighter_fights_dict[opp_opp_id]
                opp_opp_this  = opp_opp_hist[
                    opp_opp_hist['fight_id'] == opp_fight['fight_id']
                ]
                if len(opp_opp_this) > 0:
                    opp_allowed.append(opp_opp_this[stat].iloc[0])
                    opp_dates.append(opp_fight['event_date'])

        if len(opp_allowed) >= 2:
            opp_w     = time_decay_weights(opp_dates, fdate)
            opp_n_eff = kish_effective_n(opp_w)
            opp_mu    = bayesian_smooth(
                np.average(opp_allowed, weights=opp_w),
                opp_n_eff, pop_means[stat], K_MEAN
            )
            opp_sigma = max(bayesian_smooth(
                np.average(np.abs(np.array(opp_allowed) -
                    np.average(opp_allowed, weights=opp_w)), weights=opp_w),
                opp_n_eff, pop_mads[stat], K_MAD
            ), 0.01)
            snapshot = float(np.clip((smoothed - opp_mu) / opp_sigma, -7, 7))
        else:
            snapshot = 0.0

        row[f'{stat}_adjperf_snapshot'] = snapshot

    adjperf_rows.append(row)

adjperf_history_df = pd.DataFrame(adjperf_rows).sort_values('event_date').reset_index(drop=True)

# Build per-fighter dict for O(1) lookups
fighter_adjperf_history = {
    fid: grp.sort_values('event_date').reset_index(drop=True)
    for fid, grp in adjperf_history_df.groupby('fighter_id')
}

print(f"Done in {time.time()-start:.1f}s")
print(f"✓ AdjPerf history cached for {len(fighter_adjperf_history)} fighters")
print(f"✓ Total records: {len(adjperf_history_df)}")

Pre-caching per-fight AdjPerf z-scores...
Done in 330.7s
✓ AdjPerf history cached for 2661 fighters
✓ Total records: 17755


In [13]:

# ============================================================
# CELL 5: Generate Base Fights (2016–2026, clean methods only)
# ============================================================

EXCLUDED_METHODS = ['S-DEC', 'M-DEC', 'Overturned', 'CNC', 'DQ', 'Other', 'Decision']
MIN_PREV_FIGHTS  = 3   # both fighters need this many prior UFC fights

def generate_base_fights_filtered(start_date='2014-01-01', end_date='2026-12-31'):
    excl = "', '".join(EXCLUDED_METHODS)
    fights = pd.read_sql(f"""
        SELECT 
            f.fight_id,
            f.event_date,
            f.event_name,
            f.weight_class,
            f.method,
            f.ending_round,
            f.ending_time,
            ff1.fighter_id  as fighter_1_id,
            fv1.name        as fighter_1_name,
            ff2.fighter_id  as fighter_2_id,
            fv2.name        as fighter_2_name,
            CASE WHEN ff1.result = 'win' THEN 1 ELSE 0 END as winner
        FROM fights_v2 f
        JOIN fight_fighters_v2 ff1 ON f.fight_id = ff1.fight_id AND ff1.corner = 'fighter_1'
        JOIN fight_fighters_v2 ff2 ON f.fight_id = ff2.fight_id AND ff2.corner = 'fighter_2'
        JOIN fighters_v2 fv1       ON ff1.fighter_id = fv1.fighter_id
        JOIN fighters_v2 fv2       ON ff2.fighter_id = fv2.fighter_id
        WHERE f.event_date >= '{start_date}'
          AND f.event_date <= '{end_date}'
          AND f.method NOT IN ('{excl}')
        ORDER BY f.event_date
    """, conn)

    valid = []
    for _, fight in fights.iterrows():
        f1_id, f2_id, fdate = fight['fighter_1_id'], fight['fighter_2_id'], fight['event_date']

        f1_prev = len(fighter_fights_dict[f1_id][fighter_fights_dict[f1_id]['event_date'] < fdate]) \
                  if f1_id in fighter_fights_dict else 0
        f2_prev = len(fighter_fights_dict[f2_id][fighter_fights_dict[f2_id]['event_date'] < fdate]) \
                  if f2_id in fighter_fights_dict else 0

        if f1_prev >= MIN_PREV_FIGHTS and f2_prev >= MIN_PREV_FIGHTS:
            valid.append(fight)

    filtered = pd.DataFrame(valid)
    print(f"Raw fights  (2016-{end_date[:4]}): {len(fights)}")
    print(f"After {MIN_PREV_FIGHTS}+ fight filter: {len(filtered)}")
    print(f"Dropped: {len(fights)-len(filtered)} ({(len(fights)-len(filtered))/len(fights)*100:.1f}%)")
    return filtered


base_fights = generate_base_fights_filtered()
print(f"\n✔ Base fights shape: {base_fights.shape}")
print(f"✔ Date range: {base_fights['event_date'].min()} to {base_fights['event_date'].max()}")
print(f"✔ Winner distribution: {base_fights['winner'].value_counts().to_dict()}")

Raw fights  (2016-2026): 5721
After 3+ fight filter: 2837
Dropped: 2884 (50.4%)

✔ Base fights shape: (2837, 12)
✔ Date range: 2014-01-15 to 2026-02-07
✔ Winner distribution: {0: 1450, 1: 1387}


In [14]:
# ============================================================
# CELL 6: Three-Layer Feature Function
#          Changes: K_MAD=4.0, median-based MAD
# ============================================================

STATS  = ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg',
          'ctrl_time_per_min', 'kd_per_min']
K_MEAN = 4.0
K_MAD  = 4.0   # Changed from 6.0 to match author
WINDOW = 5

def bayesian_smooth(observed, n_eff, population_mean, k):
    w = n_eff / (n_eff + k)
    return w * observed + (1 - w) * population_mean

def calculate_three_layer_features_v2(fighter_id, opponent_id, as_of_date,
                                       stats=STATS, window=WINDOW):
    if fighter_id not in fighter_fights_dict or opponent_id not in fighter_fights_dict:
        return None

    fighter_hist  = fighter_fights_dict[fighter_id]
    opponent_hist = fighter_fights_dict[opponent_id]

    fighter_prev  = fighter_hist[fighter_hist['event_date']   < as_of_date]
    opponent_prev = opponent_hist[opponent_hist['event_date'] < as_of_date]

    if len(fighter_prev) == 0 or len(opponent_prev) == 0:
        return None

    pop_means = {s: float(np.median(all_fight_stats[s].values)) for s in stats}
    pop_mads  = {s: float(np.median(np.abs(
        all_fight_stats[s].values - pop_means[s]
    ))) for s in stats}

    fighter_recent  = fighter_prev.tail(window)
    fighter_weights = time_decay_weights(fighter_recent['event_date'].tolist(), as_of_date)
    fighter_n_eff   = kish_effective_n(fighter_weights)

    features = {}

    for stat in stats:
        # --- Layer 1: Fighter's time-decayed average ---
        decayed_avg   = np.average(fighter_recent[stat].values, weights=fighter_weights)
        smoothed_stat = bayesian_smooth(decayed_avg, fighter_n_eff, pop_means[stat], K_MEAN)
        features[f'{stat}_dec_avg'] = smoothed_stat

        # --- Layer 2: Opponent's allowed baseline ---
        opp_allowed, opp_dates = [], []
        for _, opp_fight in opponent_prev.iterrows():
            opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
            if opp_opp_id and opp_opp_id in fighter_fights_dict:
                opp_opp_fights     = fighter_fights_dict[opp_opp_id]
                opp_opp_this_fight = opp_opp_fights[
                    opp_opp_fights['fight_id'] == opp_fight['fight_id']
                ]
                if len(opp_opp_this_fight) > 0:
                    opp_allowed.append(opp_opp_this_fight[stat].iloc[0])
                    opp_dates.append(opp_fight['event_date'])

        if len(opp_allowed) >= 2:
            opp_weights = time_decay_weights(opp_dates, as_of_date)
            opp_n_eff   = kish_effective_n(opp_weights)
            opp_mean    = np.average(opp_allowed, weights=opp_weights)

            # CHANGED: Median-based MAD instead of weighted mean of abs devs
            opp_mad     = float(np.median(np.abs(np.array(opp_allowed) - np.median(opp_allowed))))

            opp_mu    = bayesian_smooth(opp_mean, opp_n_eff, pop_means[stat], K_MEAN)
            opp_sigma = max(
                bayesian_smooth(opp_mad, opp_n_eff, pop_mads[stat], K_MAD),
                0.01
            )
            features[f'{stat}_opp_dec_avg'] = opp_mu
            features[f'{stat}_opp_mad']     = opp_sigma
            features[f'{stat}_adjperf']     = np.clip(
                (smoothed_stat - opp_mu) / opp_sigma, -7, 7
            )
        else:
            features[f'{stat}_opp_dec_avg'] = pop_means[stat]
            features[f'{stat}_opp_mad']     = 1.0
            features[f'{stat}_adjperf']     = 0.0

        # --- dec_adjperf — decay average of historical z-scores ---
        if fighter_id in fighter_adjperf_history:
            ap_hist  = fighter_adjperf_history[fighter_id]
            ap_prev  = ap_hist[ap_hist['event_date'] < as_of_date].tail(window)
            snap_col = f'{stat}_adjperf_snapshot'

            if len(ap_prev) > 0 and snap_col in ap_prev.columns:
                ap_weights = time_decay_weights(
                    ap_prev['event_date'].tolist(), as_of_date
                )
                ap_n_eff   = kish_effective_n(ap_weights)
                ap_dec_avg = np.average(ap_prev[snap_col].values, weights=ap_weights)
                features[f'{stat}_dec_adjperf'] = bayesian_smooth(
                    ap_dec_avg, ap_n_eff, 0.0, K_MEAN
                )
            else:
                features[f'{stat}_dec_adjperf'] = 0.0
        else:
            features[f'{stat}_dec_adjperf'] = 0.0

    return features


print("✓ Three-layer feature function ready")
print(f"  STATS: {STATS}")
print(f"  K_MEAN={K_MEAN}, K_MAD={K_MAD} (was 6.0), WINDOW={WINDOW}")
print(f"  MAD method: median-based (was weighted mean)")

✓ Three-layer feature function ready
  STATS: ['slpm', 'str_acc', 'td_acc', 'td_avg', 'sub_avg', 'ctrl_time_per_min', 'kd_per_min']
  K_MEAN=4.0, K_MAD=4.0 (was 6.0), WINDOW=5
  MAD method: median-based (was weighted mean)


In [15]:
# ============================================================
# CELL 6b: Strike Breakdown Feature Function (Refactored)
# ============================================================
# NEW: Added exchange_ratio, targeting_share, strike_share
# to STRIKE_STATS_OFF — each gets dec_avg + adjperf treatment.

STRIKE_STATS_OFF = ['head_lpm', 'body_lpm', 'leg_lpm',
                    'distance_lpm', 'clinch_lpm', 'ground_lpm',
                    'head_acc', 'body_acc', 'distance_acc',
                    # NEW derived ratio features
                    'exchange_ratio', 'targeting_share', 'strike_share']

STRIKE_STATS_DEF = ['head_allowed', 'body_allowed', 'leg_allowed',
                    'distance_allowed', 'clinch_allowed', 'ground_allowed']

STRIKE_ADJPERF_OFF = STRIKE_STATS_OFF
STRIKE_ADJPERF_DEF = STRIKE_STATS_DEF


def calculate_strike_features(fighter_id, opponent_id, as_of_date, window=WINDOW):
    features = {}

    # Precompute population priors
    off_priors = {}
    for s in STRIKE_STATS_OFF:
        if s in strike_breakdown.columns:
            median = float(strike_breakdown[s].median())
            mad    = float(max(np.median(np.abs(
                strike_breakdown[s].values - median
            )), 0.01))
            off_priors[s] = {'mean': median, 'mad': mad}
        else:
            off_priors[s] = {'mean': 0.0, 'mad': 1.0}

    def_priors = {
        s: {
            'mean': float(strike_defense_df[s].median()),
            'mad':  float(max(np.median(np.abs(
                strike_defense_df[s].values - strike_defense_df[s].median()
            )), 0.01))
        } for s in STRIKE_STATS_DEF
    }

    # --- Fighter's own offensive decayed averages ---
    fighter_off_smoothed = {}
    if fighter_id in strike_breakdown_dict:
        hist = strike_breakdown_dict[fighter_id]
        prev = hist[hist['event_date'] < as_of_date].tail(window)
        if len(prev) > 0:
            weights = time_decay_weights(prev['event_date'].tolist(), as_of_date)
            n_eff   = kish_effective_n(weights)
            for s in STRIKE_STATS_OFF:
                if s not in prev.columns:
                    features[f'{s}_dec_avg'] = off_priors[s]['mean']
                    fighter_off_smoothed[s]  = off_priors[s]['mean']
                    continue
                dec_avg  = np.average(prev[s].values, weights=weights)
                smoothed = bayesian_smooth(dec_avg, n_eff, off_priors[s]['mean'], K_MEAN)
                features[f'{s}_dec_avg'] = smoothed
                fighter_off_smoothed[s]  = smoothed
        else:
            for s in STRIKE_STATS_OFF:
                features[f'{s}_dec_avg'] = off_priors[s]['mean']
                fighter_off_smoothed[s]  = off_priors[s]['mean']
    else:
        for s in STRIKE_STATS_OFF:
            features[f'{s}_dec_avg'] = off_priors[s]['mean']
            fighter_off_smoothed[s]  = off_priors[s]['mean']

    # --- Fighter's own defensive decayed averages ---
    fighter_def_smoothed = {}
    if fighter_id in strike_defense_dict:
        hist = strike_defense_dict[fighter_id]
        prev = hist[hist['event_date'] < as_of_date].tail(window)
        if len(prev) > 0:
            weights = time_decay_weights(prev['event_date'].tolist(), as_of_date)
            n_eff   = kish_effective_n(weights)
            for s in STRIKE_STATS_DEF:
                dec_avg  = np.average(prev[s].values, weights=weights)
                smoothed = bayesian_smooth(dec_avg, n_eff, def_priors[s]['mean'], K_MEAN)
                features[f'{s}_dec_avg'] = smoothed
                fighter_def_smoothed[s]  = smoothed
        else:
            for s in STRIKE_STATS_DEF:
                features[f'{s}_dec_avg'] = 0.0
                fighter_def_smoothed[s]  = 0.0
    else:
        for s in STRIKE_STATS_DEF:
            features[f'{s}_dec_avg'] = 0.0
            fighter_def_smoothed[s]  = 0.0

    # --- AdjPerf for all offensive stats ---
    if opponent_id in strike_breakdown_dict:
        opp_hist = strike_breakdown_dict[opponent_id]
        opp_prev = opp_hist[opp_hist['event_date'] < as_of_date]

        for s in STRIKE_ADJPERF_OFF:
            observed = fighter_off_smoothed[s]
            pop_mean = off_priors[s]['mean']
            pop_mad  = off_priors[s]['mad']

            opp_allowed, opp_dates = [], []
            for _, opp_fight in opp_prev.iterrows():
                opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
                if opp_opp_id and opp_opp_id in strike_breakdown_dict:
                    opp_opp_hist = strike_breakdown_dict[opp_opp_id]
                    opp_opp_this = opp_opp_hist[
                        opp_opp_hist['fight_id'] == opp_fight['fight_id']
                    ]
                    if len(opp_opp_this) > 0 and s in opp_opp_this.columns:
                        opp_allowed.append(opp_opp_this[s].iloc[0])
                        opp_dates.append(opp_fight['event_date'])

            if len(opp_allowed) >= 2:
                opp_w     = time_decay_weights(opp_dates, as_of_date)
                opp_n_eff = kish_effective_n(opp_w)
                opp_mean  = np.average(opp_allowed, weights=opp_w)
                opp_mad   = float(np.median(np.abs(
                    np.array(opp_allowed) - np.median(opp_allowed)
                )))
                mu        = bayesian_smooth(opp_mean, opp_n_eff, pop_mean, K_MEAN)
                sigma     = max(bayesian_smooth(opp_mad, opp_n_eff, pop_mad, K_MAD), 0.01)
                features[f'{s}_adjperf'] = np.clip((observed - mu) / sigma, -7, 7)
            else:
                features[f'{s}_adjperf'] = 0.0
    else:
        for s in STRIKE_ADJPERF_OFF:
            features[f'{s}_adjperf'] = 0.0

    # --- AdjPerf for all defensive stats ---
    if opponent_id in strike_breakdown_dict:
        opp_hist = strike_breakdown_dict[opponent_id]
        opp_prev = opp_hist[opp_hist['event_date'] < as_of_date]

        for s in STRIKE_ADJPERF_DEF:
            observed = fighter_def_smoothed[s]
            pop_mean = def_priors[s]['mean']
            pop_mad  = def_priors[s]['mad']

            opp_allowed, opp_dates = [], []
            for _, opp_fight in opp_prev.iterrows():
                opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
                if opp_opp_id and opp_opp_id in strike_defense_dict:
                    opp_opp_hist = strike_defense_dict[opp_opp_id]
                    opp_opp_this = opp_opp_hist[
                        opp_opp_hist['fight_id'] == opp_fight['fight_id']
                    ]
                    if len(opp_opp_this) > 0:
                        opp_allowed.append(opp_opp_this[s].iloc[0])
                        opp_dates.append(opp_fight['event_date'])

            if len(opp_allowed) >= 2:
                opp_w     = time_decay_weights(opp_dates, as_of_date)
                opp_n_eff = kish_effective_n(opp_w)
                opp_mean  = np.average(opp_allowed, weights=opp_w)
                opp_mad   = float(np.median(np.abs(
                    np.array(opp_allowed) - np.median(opp_allowed)
                )))
                mu        = bayesian_smooth(opp_mean, opp_n_eff, pop_mean, K_MEAN)
                sigma     = max(bayesian_smooth(opp_mad, opp_n_eff, pop_mad, K_MAD), 0.01)
                features[f'{s}_adjperf'] = np.clip((observed - mu) / sigma, -7, 7)
            else:
                features[f'{s}_adjperf'] = 0.0
    else:
        for s in STRIKE_ADJPERF_DEF:
            features[f'{s}_adjperf'] = 0.0

    return features


print("✓ Strike feature function ready (with derived ratios)")
print(f"  Offensive: {len(STRIKE_STATS_OFF)} stats × 2 (dec_avg + adjperf)")
print(f"  NEW: exchange_ratio, targeting_share, strike_share")
print(f"  Defensive: {len(STRIKE_STATS_DEF)} stats × 2 (dec_avg + adjperf)")
print(f"  NOTE: AdjPerf now uses median-based MAD (consistent with Cell 6)")

✓ Strike feature function ready (with derived ratios)
  Offensive: 12 stats × 2 (dec_avg + adjperf)
  NEW: exchange_ratio, targeting_share, strike_share
  Defensive: 6 stats × 2 (dec_avg + adjperf)
  NOTE: AdjPerf now uses median-based MAD (consistent with Cell 6)


In [16]:
# ============================================================
# CELL 6c: Rolling Career Stats (Refactored)
# ============================================================
# NEW: Added decision_rate (BB tau=20) and sub_landing_rate (BB tau=9)
# Both are Beta-Binomial smoothed with weight-class priors.

# ── Pre-compute weight-class priors for career stats ──────
career_wc_priors = {}
career_global_priors = {}

# Build from fighter_results_dict + all_fight_stats
all_results_list = []
for (fid, fight_id), result in fighter_results_dict.items():
    # Look up method for decision rate
    method = None
    fight_rows = all_fight_stats[all_fight_stats['fight_id'] == fight_id]
    if len(fight_rows) > 0:
        method = fight_rows.iloc[0].get('method', None)

    all_results_list.append({
        'fighter_id': fid,
        'fight_id': fight_id,
        'is_win': 1 if result in ('win', 'ko_win') else 0,
        'is_ko':  1 if result == 'ko_win' else 0,
        'is_decision': 1 if method and 'DEC' in str(method).upper() else 0,
    })
results_df = pd.DataFrame(all_results_list)

# Merge weight class
results_with_wc = results_df.merge(
    all_fight_stats[['fight_id', 'fighter_id', 'weight_class']].drop_duplicates(),
    on=['fight_id', 'fighter_id'], how='left'
)
results_with_wc['wc_norm'] = results_with_wc['weight_class'].apply(normalize_weight_class)

# Global priors
career_global_priors['win'] = results_with_wc['is_win'].mean()
career_global_priors['ko']  = results_with_wc['is_ko'].mean()
career_global_priors['dec'] = results_with_wc['is_decision'].mean()

# Per weight class
for wc, grp in results_with_wc[results_with_wc['wc_norm'].notna()].groupby('wc_norm'):
    career_wc_priors[wc] = {
        'win': grp['is_win'].mean(),
        'ko':  grp['is_ko'].mean(),
        'dec': grp['is_decision'].mean(),
    }

# ── Pre-cache decision results per fight for O(1) lookups ──
# Map (fighter_id, fight_id) → is_decision
fight_decision_dict = {}
for _, row in results_with_wc.iterrows():
    fight_decision_dict[(row['fighter_id'], row['fight_id'])] = row['is_decision']

# Tau values from author
TAU_WIN = 25.0
TAU_KO  = 23.0
TAU_DEC = 20.0  # NEW
TAU_SUB_LAND = 9.0  # NEW

print(f"✔ Career priors computed")
print(f"  Global win rate:      {career_global_priors['win']:.3f}")
print(f"  Global KO rate:       {career_global_priors['ko']:.3f}")
print(f"  Global decision rate: {career_global_priors['dec']:.3f}")


def get_career_wc_prior(wc, stat):
    wc_n = normalize_weight_class(wc) if wc else None
    if wc_n and wc_n in career_wc_priors and stat in career_wc_priors[wc_n]:
        return career_wc_priors[wc_n][stat]
    return career_global_priors.get(stat, 0.5)


def calculate_career_stats(fighter_id, opponent_id, fight_id, as_of_date, window=10):
    features = {}

    defaults = {
        'days_since_last_fight': 180,
        'win_ratio':             0.5,
        'win_adjperf':           0.0,
        'ko_rate':               0.0,
        'ko_opp_dec_avg':        0.0,
        'decision_rate':         0.5,
        'sub_landing_rate':      0.0,
        'td_defense':            0.5,
        'td_land_ratio_opp':     0.0,
        'ctrl_ratio_opp':        0.0,
        'sub_att_allowed_pm':    0.0,
        'kd_allowed_pm':         0.0,
    }

    if fighter_id not in fighter_fights_dict:
        return defaults

    hist = fighter_fights_dict[fighter_id]
    prev = hist[hist['event_date'] < as_of_date]

    if len(prev) == 0:
        return defaults

    # --- Days since last fight ---
    last_date = prev.iloc[-1]['event_date']
    features['days_since_last_fight'] = (
        datetime.strptime(as_of_date, "%Y-%m-%d") -
        datetime.strptime(last_date,  "%Y-%m-%d")
    ).days

    # --- Get fight weight class for priors ---
    fight_wc = None
    if fight_id in all_fight_stats['fight_id'].values:
        wc_row = all_fight_stats[all_fight_stats['fight_id'] == fight_id]
        if len(wc_row) > 0:
            fight_wc = wc_row.iloc[0]['weight_class']

    # --- Win ratio + KO rate + Decision rate (Beta-Binomial smoothed) ---
    prior_fids = prev['fight_id'].tolist()
    results    = [fighter_results_dict.get((fighter_id, fid)) for fid in prior_fids]
    results    = [r for r in results if r is not None]

    n_fights = len(results)
    if n_fights > 0:
        n_wins = sum(1 for r in results if r in ('win', 'ko_win'))
        n_kos  = sum(1 for r in results if r == 'ko_win')

        # Beta-Binomial smoothed win ratio
        win_prior = get_career_wc_prior(fight_wc, 'win')
        win_ratio = (win_prior * TAU_WIN + n_wins) / (TAU_WIN + n_fights)

        # Beta-Binomial smoothed KO rate
        ko_prior = get_career_wc_prior(fight_wc, 'ko')
        ko_rate  = (ko_prior * TAU_KO + n_kos) / (TAU_KO + n_fights)

        # NEW: Beta-Binomial smoothed decision rate
        n_decs   = sum(fight_decision_dict.get((fighter_id, fid), 0) for fid in prior_fids)
        dec_prior = get_career_wc_prior(fight_wc, 'dec')
        decision_rate = (dec_prior * TAU_DEC + n_decs) / (TAU_DEC + n_fights)

        features['win_ratio']     = win_ratio
        features['ko_rate']       = ko_rate
        features['decision_rate'] = decision_rate
    else:
        features['win_ratio']     = get_career_wc_prior(fight_wc, 'win')
        features['ko_rate']       = get_career_wc_prior(fight_wc, 'ko')
        features['decision_rate'] = get_career_wc_prior(fight_wc, 'dec')

    win_ratio = features['win_ratio']

    # --- NEW: Sub landing rate (Beta-Binomial smoothed) ---
    # sub_landed / sub_attempted across career
    total_sub_landed   = 0
    total_sub_attempted = 0
    for _, fight_row in prev.iterrows():
        result = fighter_results_dict.get((fighter_id, fight_row['fight_id']))
        sub_att = fight_row.get('sub_attempts', 0)
        total_sub_attempted += sub_att
        # A sub is "landed" if this fighter won by submission in this fight
        if result == 'win':
            method = None
            fight_rows = all_fight_stats[
                all_fight_stats['fight_id'] == fight_row['fight_id']
            ]
            if len(fight_rows) > 0:
                method = fight_rows.iloc[0].get('method', '')
            if method and 'SUB' in str(method).upper():
                total_sub_landed += 1

    # Use global sub success rate as prior (~5-10% of attempts)
    sub_land_prior = 0.05  # rough global prior
    if total_sub_attempted > 0:
        features['sub_landing_rate'] = (
            (sub_land_prior * TAU_SUB_LAND + total_sub_landed) /
            (TAU_SUB_LAND + total_sub_attempted)
        )
    else:
        features['sub_landing_rate'] = sub_land_prior

    # --- Win rate AdjPerf ---
    if opponent_id in fighter_fights_dict:
        opp_hist = fighter_fights_dict[opponent_id]
        opp_prev = opp_hist[opp_hist['event_date'] < as_of_date]

        opp_win_rates, opp_dates = [], []
        for _, opp_fight in opp_prev.iterrows():
            opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
            if opp_opp_id:
                opp_opp_fids = fighter_fights_dict.get(opp_opp_id)
                if opp_opp_fids is not None:
                    opp_opp_prev = opp_opp_fids[
                        opp_opp_fids['event_date'] < opp_fight['event_date']
                    ]
                    opp_opp_results = [
                        fighter_results_dict.get((opp_opp_id, fid))
                        for fid in opp_opp_prev['fight_id'].tolist()
                    ]
                    opp_opp_results = [r for r in opp_opp_results if r is not None]
                    if len(opp_opp_results) > 0:
                        opp_n = len(opp_opp_results)
                        opp_wins = sum(1 for r in opp_opp_results if r in ('win', 'ko_win'))
                        opp_opp_wr = (get_career_wc_prior(fight_wc, 'win') * TAU_WIN + opp_wins) / (TAU_WIN + opp_n)
                        opp_win_rates.append(opp_opp_wr)
                        opp_dates.append(opp_fight['event_date'])

        if len(opp_win_rates) >= 2:
            opp_w     = time_decay_weights(opp_dates, as_of_date)
            opp_n_eff = kish_effective_n(opp_w)
            opp_mean  = np.average(opp_win_rates, weights=opp_w)
            opp_mad   = float(np.median(np.abs(np.array(opp_win_rates) - np.median(opp_win_rates))))
            pop_mean  = 0.5
            pop_mad   = 0.15
            mu        = bayesian_smooth(opp_mean, opp_n_eff, pop_mean, K_MEAN)
            sigma     = max(bayesian_smooth(opp_mad, opp_n_eff, pop_mad, K_MAD), 0.01)
            features['win_adjperf'] = np.clip((win_ratio - mu) / sigma, -7, 7)
        else:
            features['win_adjperf'] = 0.0
    else:
        features['win_adjperf'] = 0.0

    # --- Opponent stats (rolling window) ---
    recent = prev.tail(window)
    total_td_att, total_td_land   = 0, 0
    total_sub_att, total_minutes  = 0, 0
    total_kd, total_minutes_kd    = 0, 0
    total_ctrl, total_fight_mins  = 0, 0
    opp_ko_rates, opp_ko_dates    = [], []
    opp_td_ratios, opp_td_dates   = [], []

    for _, fight_row in recent.iterrows():
        opp_id = opponents_dict.get((fight_row['fight_id'], fighter_id))
        if opp_id and opp_id in fighter_fights_dict:
            opp_hist  = fighter_fights_dict[opp_id]
            opp_this  = opp_hist[opp_hist['fight_id'] == fight_row['fight_id']]
            if len(opp_this) > 0:
                opp_row = opp_this.iloc[0]

                total_td_att  += opp_row['td_attempted']
                total_td_land += opp_row['td_landed']
                total_sub_att += opp_row['sub_attempts']
                total_minutes += opp_row['actual_minutes']
                total_kd         += opp_row['kd_per_min'] * opp_row['actual_minutes']
                total_minutes_kd += opp_row['actual_minutes']
                total_ctrl       += opp_row['ctrl_time_per_min'] * opp_row['actual_minutes']
                total_fight_mins += opp_row['actual_minutes']

                # KO rate opponent — smoothed
                opp_prev_fights = opp_hist[opp_hist['event_date'] < fight_row['event_date']]
                opp_results = [
                    fighter_results_dict.get((opp_id, fid))
                    for fid in opp_prev_fights['fight_id'].tolist()
                ]
                opp_results = [r for r in opp_results if r is not None]
                if len(opp_results) > 0:
                    opp_n_ko = len(opp_results)
                    opp_kos  = sum(1 for r in opp_results if r == 'ko_win')
                    opp_ko_smoothed = (get_career_wc_prior(fight_wc, 'ko') * TAU_KO + opp_kos) / (TAU_KO + opp_n_ko)
                    opp_ko_rates.append(opp_ko_smoothed)
                    opp_ko_dates.append(fight_row['event_date'])

                if opp_row['td_attempted'] > 0:
                    opp_td_ratios.append(opp_row['td_landed'] / opp_row['td_attempted'])
                    opp_td_dates.append(fight_row['event_date'])

    features['td_defense'] = (
        1 - (total_td_land / total_td_att) if total_td_att > 0 else 0.5
    )
    features['sub_att_allowed_pm'] = (
        total_sub_att / total_minutes if total_minutes > 0 else 0.0
    )
    features['kd_allowed_pm'] = (
        total_kd / total_minutes_kd if total_minutes_kd > 0 else 0.0
    )
    features['ctrl_ratio_opp'] = (
        total_ctrl / total_fight_mins if total_fight_mins > 0 else 0.0
    )

    if len(opp_ko_rates) >= 2:
        opp_w = time_decay_weights(opp_ko_dates, as_of_date)
        features['ko_opp_dec_avg'] = float(np.average(opp_ko_rates, weights=opp_w))
    else:
        features['ko_opp_dec_avg'] = 0.0

    if len(opp_td_ratios) >= 2:
        opp_w = time_decay_weights(opp_td_dates, as_of_date)
        features['td_land_ratio_opp'] = float(np.average(opp_td_ratios, weights=opp_w))
    else:
        features['td_land_ratio_opp'] = 0.0

    return features


print("✔ Career stats function ready (with decision_rate + sub_landing_rate)")
print(f"  win_ratio: tau={TAU_WIN}, ko_rate: tau={TAU_KO}")
print(f"  NEW decision_rate: tau={TAU_DEC}")
print(f"  NEW sub_landing_rate: tau={TAU_SUB_LAND}")

✔ Career priors computed
  Global win rate:      0.491
  Global KO rate:       0.169
  Global decision rate: 0.426
✔ Career stats function ready (with decision_rate + sub_landing_rate)
  win_ratio: tau=25.0, ko_rate: tau=23.0
  NEW decision_rate: tau=20.0
  NEW sub_landing_rate: tau=9.0


In [17]:
# ============================================================
# CELL 6d: Round 1 Feature Function (Refactored)
# ============================================================
# NEW: Added r1_head_lpm, r1_body_lpm, r1_leg_lpm, r1_clinch_lpm
# Each gets dec_avg + adjperf + opp_dec_avg treatment.

R1_STATS = ['r1_slpm', 'r1_ctrl_per_min', 'r1_td_acc',
            'r1_kd_per_min', 'r1_rev_per_min', 'r1_td_att_per_min',
            # NEW: R1 strike breakdown
            'r1_head_lpm', 'r1_body_lpm', 'r1_leg_lpm', 'r1_clinch_lpm']

def calculate_r1_features(fighter_id, opponent_id, as_of_date, window=WINDOW):
    features = {}

    # Precompute population priors
    r1_priors = {}
    for s in R1_STATS:
        if s in r1_stats.columns:
            median = float(r1_stats[s].median())
            mad    = float(max(np.median(np.abs(r1_stats[s].values - median)), 0.01))
            r1_priors[s] = {'mean': median, 'mad': mad}
        else:
            r1_priors[s] = {'mean': 0.0, 'mad': 1.0}

    # Ensure r1_td_att_per_min exists
    if 'r1_td_att_per_min' not in r1_stats.columns:
        r1_stats['r1_td_att_per_min'] = r1_stats['r1_td_attempted'] / r1_stats['r1_minutes'].clip(lower=0.1)
        r1_stats['r1_td_att_per_min'] = r1_stats['r1_td_att_per_min'].replace(
            [np.inf, -np.inf], np.nan
        ).fillna(0)
        for fid in r1_stats_dict:
            r1_stats_dict[fid] = r1_stats[r1_stats['fighter_id'] == fid].sort_values(
                'event_date'
            ).reset_index(drop=True)
        median = float(r1_stats['r1_td_att_per_min'].median())
        mad    = float(max(np.median(np.abs(
            r1_stats['r1_td_att_per_min'].values - median
        )), 0.01))
        r1_priors['r1_td_att_per_min'] = {'mean': median, 'mad': mad}

    # --- Fighter's own R1 decayed averages ---
    fighter_smoothed = {}
    if fighter_id in r1_stats_dict:
        hist = r1_stats_dict[fighter_id]
        prev = hist[hist['event_date'] < as_of_date].tail(window)
        if len(prev) > 0:
            weights = time_decay_weights(prev['event_date'].tolist(), as_of_date)
            n_eff   = kish_effective_n(weights)
            for s in R1_STATS:
                if s not in prev.columns:
                    features[f'{s}_dec_avg'] = r1_priors[s]['mean']
                    fighter_smoothed[s]      = r1_priors[s]['mean']
                    continue
                dec_avg  = np.average(prev[s].values, weights=weights)
                smoothed = bayesian_smooth(dec_avg, n_eff, r1_priors[s]['mean'], K_MEAN)
                features[f'{s}_dec_avg'] = smoothed
                fighter_smoothed[s]      = smoothed
        else:
            for s in R1_STATS:
                features[f'{s}_dec_avg'] = r1_priors[s]['mean']
                fighter_smoothed[s]      = r1_priors[s]['mean']
    else:
        for s in R1_STATS:
            features[f'{s}_dec_avg'] = r1_priors[s]['mean']
            fighter_smoothed[s]      = r1_priors[s]['mean']

    # --- Full AdjPerf + opponent allowed for all R1 stats ---
    if opponent_id in r1_stats_dict:
        opp_hist = r1_stats_dict[opponent_id]
        opp_prev = opp_hist[opp_hist['event_date'] < as_of_date]

        for s in R1_STATS:
            observed = fighter_smoothed[s]
            pop_mean = r1_priors[s]['mean']
            pop_mad  = r1_priors[s]['mad']

            opp_allowed, opp_dates = [], []
            for _, opp_fight in opp_prev.iterrows():
                opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
                if opp_opp_id and opp_opp_id in r1_stats_dict:
                    opp_opp_hist = r1_stats_dict[opp_opp_id]
                    opp_opp_this = opp_opp_hist[
                        opp_opp_hist['fight_id'] == opp_fight['fight_id']
                    ]
                    if len(opp_opp_this) > 0 and s in opp_opp_this.columns:
                        opp_allowed.append(opp_opp_this[s].iloc[0])
                        opp_dates.append(opp_fight['event_date'])

            if len(opp_allowed) >= 2:
                opp_w     = time_decay_weights(opp_dates, as_of_date)
                opp_n_eff = kish_effective_n(opp_w)
                opp_mean  = np.average(opp_allowed, weights=opp_w)
                opp_mad   = float(np.median(np.abs(
                    np.array(opp_allowed) - np.median(opp_allowed)
                )))
                mu        = bayesian_smooth(opp_mean, opp_n_eff, pop_mean, K_MEAN)
                sigma     = max(bayesian_smooth(opp_mad, opp_n_eff, pop_mad, K_MAD), 0.01)
                features[f'{s}_adjperf']     = np.clip((observed - mu) / sigma, -7, 7)
                features[f'{s}_opp_dec_avg'] = mu
            else:
                features[f'{s}_adjperf']     = 0.0
                features[f'{s}_opp_dec_avg'] = pop_mean
    else:
        for s in R1_STATS:
            features[f'{s}_adjperf']     = 0.0
            features[f'{s}_opp_dec_avg'] = r1_priors[s]['mean']

    # --- Leg landing R1 opponent allowed ---
    leg_opp_allowed, leg_opp_dates = [], []
    if opponent_id in r1_stats_dict:
        opp_prev = r1_stats_dict[opponent_id]
        opp_prev = opp_prev[opp_prev['event_date'] < as_of_date]
        for _, opp_fight in opp_prev.iterrows():
            opp_opp_id = opponents_dict.get((opp_fight['fight_id'], opponent_id))
            if opp_opp_id and opp_opp_id in strike_breakdown_dict:
                opp_opp_strikes = strike_breakdown_dict[opp_opp_id]
                opp_opp_this    = opp_opp_strikes[
                    opp_opp_strikes['fight_id'] == opp_fight['fight_id']
                ]
                if len(opp_opp_this) > 0:
                    leg_opp_allowed.append(opp_opp_this['leg_lpm'].iloc[0])
                    leg_opp_dates.append(opp_fight['event_date'])

    if len(leg_opp_allowed) >= 2:
        opp_w = time_decay_weights(leg_opp_dates, as_of_date)
        features['leg_land_r1_opp_dec_avg'] = float(np.average(leg_opp_allowed, weights=opp_w))
    else:
        features['leg_land_r1_opp_dec_avg'] = 0.0

    return features


print("✔ R1 feature function ready (with strike breakdown)")
print(f"  R1 stats: {len(R1_STATS)} — each gets dec_avg + adjperf + opp_dec_avg")
print(f"  NEW: r1_head_lpm, r1_body_lpm, r1_leg_lpm, r1_clinch_lpm")
print(f"  Plus: leg_land_r1_opp_dec_avg")

✔ R1 feature function ready (with strike breakdown)
  R1 stats: 10 — each gets dec_avg + adjperf + opp_dec_avg
  NEW: r1_head_lpm, r1_body_lpm, r1_leg_lpm, r1_clinch_lpm
  Plus: leg_land_r1_opp_dec_avg


In [18]:
# ============================================================
# CELL 7: Generate Features for All Fights
# ============================================================

def generate_features(base_df):
    rows  = []
    total = len(base_df)

    for idx, fight in base_df.iterrows():
        if idx % 500 == 0:
            print(f"  Processing {idx}/{total}...")

        f1_id = fight['fighter_1_id']
        f2_id = fight['fighter_2_id']
        fdate = fight['event_date']
        fid   = fight['fight_id']


        # Replace with:
        f1_feats = calculate_three_layer_features_v2(f1_id, f2_id, fdate)
        f2_feats = calculate_three_layer_features_v2(f2_id, f1_id, fdate)

        if f1_feats is None or f2_feats is None:
            continue

        # --- Strike breakdown features ---
        f1_feats.update(calculate_strike_features(f1_id, f2_id, fdate))
        f2_feats.update(calculate_strike_features(f2_id, f1_id, fdate))

        # --- Round 1 features ---
        f1_feats.update(calculate_r1_features(f1_id, f2_id, fdate))
        f2_feats.update(calculate_r1_features(f2_id, f1_id, fdate))

        # --- Rolling career stats ---
        f1_feats.update(calculate_career_stats(f1_id, f2_id, fid, fdate))
        f2_feats.update(calculate_career_stats(f2_id, f1_id, fid, fdate))
        # --- Assemble row ---
        row = {'fight_id': fid}
        for k, v in f1_feats.items(): row[f'f1_{k}'] = v
        for k, v in f2_feats.items(): row[f'f2_{k}'] = v
        for k   in f1_feats.keys():   row[f'diff_{k}'] = row[f'f1_{k}'] - row[f'f2_{k}']
        rows.append(row)

    feats_df = pd.DataFrame(rows)
    return base_df.merge(feats_df, on='fight_id', how='inner')


print("Generating three-layer features...")
start = time.time()
fight_features = generate_features(base_fights)
print(f"Done in {time.time()-start:.1f}s")
print(f"✔ Shape: {fight_features.shape}")
print(f"✔ Features per fighter: {len([c for c in fight_features.columns if c.startswith('f1_')])}")

Generating three-layer features...
  Processing 500/2837...
  Processing 1000/2837...
  Processing 1500/2837...
  Processing 2000/2837...
  Processing 3000/2837...
  Processing 4000/2837...
  Processing 4500/2837...
Done in 988.7s
✔ Shape: (2837, 354)
✔ Features per fighter: 114


In [19]:
# ============================================================
# CELL 8: Physical Features & Experience
# ============================================================

def add_physical_and_experience(df, conn):
    all_ids  = set(df['fighter_1_id'].unique()) | set(df['fighter_2_id'].unique())
    fighters = pd.read_sql(f"""
        SELECT fighter_id, reach, height, dob
        FROM fighters_v2
        WHERE fighter_id IN ({','.join(f"'{fid}'" for fid in all_ids)})
    """, conn)

    fighters['reach_in']  = fighters['reach'].apply(parse_reach)
    fighters['height_in'] = fighters['height'].apply(parse_height)

    for prefix, fid_col in [('f1', 'fighter_1_id'), ('f2', 'fighter_2_id')]:
        df = df.merge(
            fighters[['fighter_id', 'reach_in', 'height_in', 'dob']],
            left_on=fid_col, right_on='fighter_id', how='left'
        ).drop('fighter_id', axis=1).rename(columns={
            'reach_in': f'{prefix}_reach', 'height_in': f'{prefix}_height', 'dob': f'{prefix}_dob'
        })

    df['f1_age'] = df.apply(lambda r: parse_age(r['f1_dob'], r['event_date']), axis=1)
    df['f2_age'] = df.apply(lambda r: parse_age(r['f2_dob'], r['event_date']), axis=1)

    df['age_diff']    = df['f1_age']    - df['f2_age']
    df['reach_diff']  = df['f1_reach']  - df['f2_reach']
    df['height_diff'] = df['f1_height'] - df['f2_height']
    df['age_ratio']   = df['f1_age']    / df['f2_age']
    df['reach_ratio'] = df['f1_reach']  / df['f2_reach']
    df['height_ratio']= df['f1_height'] / df['f2_height']

    for prefix, fid_col in [('f1', 'fighter_1_id'), ('f2', 'fighter_2_id')]:
        ufc_ages = []
        for _, fight in df.iterrows():
            fid = fight[fid_col]
            if fid in fighter_fights_dict:
                first = fighter_fights_dict[fid].iloc[0]['event_date']
                days  = (datetime.strptime(fight['event_date'], "%Y-%m-%d") -
                         datetime.strptime(first,              "%Y-%m-%d")).days
                ufc_ages.append(days / 365.25)
            else:
                ufc_ages.append(0)
        df[f'{prefix}_ufc_age'] = ufc_ages

    df['diff_ufc_age'] = df['f1_ufc_age'] - df['f2_ufc_age']
    df = df.drop(['f1_dob', 'f2_dob'], axis=1)
    return df


fight_features = add_physical_and_experience(fight_features, conn)
print(f"✔ Shape with physical features: {fight_features.shape}")

✔ Shape with physical features: (2837, 369)


In [20]:
# ============================================================
# CELL 8b: Interaction Features
# ============================================================
# Motivated by disagreement analysis:
# - Correct upsets had diff_td_avg_dec_avg of +0.348 vs +0.083
# - Correct upsets had age_diff of +0.082 vs +0.910
# These interactions help the model learn that takedown and
# striking advantages mean MORE for younger fighters

# ── Age × Takedown ──────────────────────────────────────────
# Captures: young fighter with takedown advantage over older opponent
fight_features['age_td_interaction'] = (
    fight_features['age_diff'] *
    fight_features['diff_td_avg_dec_avg']
)

# AdjPerf version — opponent-adjusted takedown advantage × age
fight_features['age_td_adjperf_interaction'] = (
    fight_features['age_diff'] *
    fight_features['diff_td_avg_adjperf']
)

# ── Age × Striking ──────────────────────────────────────────
# Captures: young fighter with striking accuracy advantage
fight_features['age_str_interaction'] = (
    fight_features['age_diff'] *
    fight_features['diff_str_acc_dec_avg']
)

# ── Age × Win Ratio ─────────────────────────────────────────
# Captures: younger fighter with better win record
fight_features['age_winratio_interaction'] = (
    fight_features['age_diff'] *
    fight_features['diff_win_ratio']
)

# ── UFC Experience × Takedown ───────────────────────────────
# Captures: less experienced fighter compensating with grappling
fight_features['ufc_age_td_interaction'] = (
    fight_features['diff_ufc_age'] *
    fight_features['diff_td_avg_dec_avg']
)

# ── UFC Experience × Striking ───────────────────────────────
# Captures: less experienced fighter with striking edge
fight_features['ufc_age_str_interaction'] = (
    fight_features['diff_ufc_age'] *
    fight_features['diff_str_acc_dec_avg']
)

# ── Takedown × Control ──────────────────────────────────────
# Captures: fighters who take down AND control — complete grapplers
fight_features['td_ctrl_interaction'] = (
    fight_features['diff_td_avg_dec_avg'] *
    fight_features['diff_ctrl_time_per_min_dec_avg']
)

# ── Striking × Head Defense ─────────────────────────────────
# Captures: fighters who land more AND absorb less — striking dominance
fight_features['str_headdef_interaction'] = (
    fight_features['diff_str_acc_dec_avg'] *
    fight_features['diff_head_allowed_dec_avg']
)

new_cols = [
    'age_td_interaction', 'age_td_adjperf_interaction',
    'age_str_interaction', 'age_winratio_interaction',
    'ufc_age_td_interaction', 'ufc_age_str_interaction',
    'td_ctrl_interaction', 'str_headdef_interaction'
]

# Fill any NaNs from multiplication
fight_features[new_cols] = fight_features[new_cols].fillna(0)

print(f"✓ Added {len(new_cols)} interaction features")
print(f"✓ fight_features shape: {fight_features.shape}")
for col in new_cols:
    print(f"  {col}: mean={fight_features[col].mean():.3f}, "
          f"std={fight_features[col].std():.3f}")

✓ Added 8 interaction features
✓ fight_features shape: (2837, 377)
  age_td_interaction: mean=-0.932, std=8.241
  age_td_adjperf_interaction: mean=-0.586, std=7.626
  age_str_interaction: mean=-0.029, std=0.305
  age_winratio_interaction: mean=-0.005, std=0.248
  ufc_age_td_interaction: mean=-0.722, std=7.444
  ufc_age_str_interaction: mean=-0.035, std=0.269
  td_ctrl_interaction: mean=0.279, std=0.604
  str_headdef_interaction: mean=-0.008, std=0.055


In [21]:
# ============================================================
# CELL 9: Optuna Hyperparameter Search + Feature Trimming
# ============================================================

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- Step 1: Train/Val split ---
train = fight_features[fight_features['event_date'] < '2024-01-01']
val   = fight_features[fight_features['event_date'] >= '2024-01-01']

FEATURE_COLS = [c for c in fight_features.columns 
                if c.startswith('diff_')
                or c in ['age_diff', 'age_ratio', 'reach_diff',
                         'height_diff', 'reach_ratio', 'height_ratio']]

X_train = train[FEATURE_COLS].fillna(0)
y_train = train['winner']
X_val   = val[FEATURE_COLS].fillna(0)
y_val   = val['winner']

print(f"✓ Train: {X_train.shape}, Val: {X_val.shape}")

# --- Step 2: Feature trimming (use fixed baseline config) ---
model_for_imp = XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    min_child_weight=15, subsample=0.6, colsample_bytree=0.5,
    gamma=2.0, reg_alpha=1.0, reg_lambda=2.0, random_state=42
)
model_for_imp.fit(X_train, y_train)

feat_imp = pd.Series(
    model_for_imp.feature_importances_, index=X_train.columns
).sort_values(ascending=False)

threshold = 0.007
low_imp = feat_imp[feat_imp < threshold].index.tolist()
X_train_trimmed = X_train.drop(columns=low_imp)
X_val_trimmed   = X_val.drop(columns=low_imp)
print(f"✓ Trimmed to {X_train_trimmed.shape[1]} features (dropped {len(low_imp)})")

# --- Step 3: Optuna search ---
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 50, 500),
        'max_depth':        trial.suggest_int('max_depth', 2, 6),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 50),
        'subsample':        trial.suggest_float('subsample', 0.4, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.8),
        'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 5.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 5.0),
        'random_state':     42,
    }
    
    m = XGBClassifier(**params)
    m.fit(X_train_trimmed, y_train)
    
    val_acc   = m.score(X_val_trimmed, y_val)
    train_acc = m.score(X_train_trimmed, y_train)
    gap = train_acc - val_acc
    
    # Penalize large train/val gaps to avoid overfitting configs
    if gap > 0.12:
        return val_acc - (gap - 0.12) * 0.5
    
    return val_acc

print(f"\n✓ Starting Optuna search (200 trials)...")
start = time.time()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200)

print(f"Done in {time.time()-start:.1f}s")

# --- Step 4: Results ---
best = study.best_trial
print(f"\n{'='*55}")
print(f"BEST TRIAL: #{best.number}")
print(f"{'='*55}")
print(f"Validation accuracy: {best.value:.4f} ({best.value*100:.1f}%)")
print(f"\nBest parameters:")
for k, v in best.params.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

# --- Step 5: Retrain with best params and report ---
best_model = XGBClassifier(**best.params, random_state=42)
best_model.fit(X_train_trimmed, y_train)

train_acc = best_model.score(X_train_trimmed, y_train)
val_acc   = best_model.score(X_val_trimmed, y_val)

print(f"\n{'='*55}")
print(f"FINAL RESULTS (best params)")
print(f"{'='*55}")
print(f"Training:   {train_acc*100:.1f}%")
print(f"Validation: {val_acc*100:.1f}%")
print(f"Gap:        {(train_acc-val_acc)*100:.1f}%")

# --- Step 6: Compare to baseline ---
print(f"\n{'='*55}")
print(f"COMPARISON TO BASELINE")
print(f"{'='*55}")
print(f"Baseline:  65.8% val, 74.4% train, 8.6% gap")
print(f"Optuna:    {val_acc*100:.1f}% val, {train_acc*100:.1f}% train, "
      f"{(train_acc-val_acc)*100:.1f}% gap")
print(f"Change:    {(val_acc-0.658)*100:+.1f}%")

# --- Step 7: Top 5 trials ---
print(f"\n{'='*55}")
print(f"TOP 5 TRIALS")
print(f"{'='*55}")
trials_sorted = sorted(study.trials, key=lambda t: t.value, reverse=True)
for t in trials_sorted[:5]:
    m = XGBClassifier(**t.params, random_state=42)
    m.fit(X_train_trimmed, y_train)
    ta = m.score(X_train_trimmed, y_train)
    print(f"  Trial {t.number:>3}: val={t.value:.4f} train={ta:.4f} "
          f"gap={ta-t.value:.4f}")

# Store for later use
model_trimmed = best_model

✓ Train: (2220, 121), Val: (617, 121)


C:\Users\Sarthak\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Trimmed to 92 features (dropped 29)

✓ Starting Optuna search (200 trials)...
Done in 138.6s

BEST TRIAL: #82
Validation accuracy: 0.6888 (68.9%)

Best parameters:
  n_estimators: 297
  max_depth: 4
  learning_rate: 0.0117
  min_child_weight: 44
  subsample: 0.6313
  colsample_bytree: 0.4470
  gamma: 4.7136
  reg_alpha: 3.1348
  reg_lambda: 0.6134

FINAL RESULTS (best params)
Training:   68.7%
Validation: 68.9%
Gap:        -0.2%

COMPARISON TO BASELINE
Baseline:  65.8% val, 74.4% train, 8.6% gap
Optuna:    68.9% val, 68.7% train, -0.2% gap
Change:    +3.1%

TOP 5 TRIALS
  Trial  82: val=0.6888 train=0.6869 gap=-0.0019
  Trial 186: val=0.6888 train=0.7032 gap=0.0143
  Trial 190: val=0.6888 train=0.7068 gap=0.0179
  Trial 191: val=0.6888 train=0.6923 gap=0.0035
  Trial 153: val=0.6872 train=0.6973 gap=0.0101


In [22]:
import pickle

pickle.dump(best_model, open('xgb_69_model.pkl', 'wb'))
pickle.dump(X_val_trimmed.columns.tolist(), open('feature_cols_69.pkl', 'wb'))
fight_features.to_parquet('fight_features_69.parquet')
print(f"✔ Saved. Val: {best_model.score(X_val_trimmed, y_val):.1%}")
print(f"   Train: {best_model.score(X_train_trimmed, y_train):.1%}")
print(f"   Features: {X_val_trimmed.shape[1]}")

# Also save the exact params so you can recreate it
import json
json.dump(best.params, open('optuna_params_69.json', 'w'), indent=2)
print(f"✔ Params saved to optuna_params_69.json")

✔ Saved. Val: 68.9%
   Train: 68.7%
   Features: 92
✔ Params saved to optuna_params_69.json


In [23]:
import pickle

# Save Optuna best model
best_model = XGBClassifier(
    n_estimators=399, max_depth=4, learning_rate=0.0315,
    min_child_weight=47, subsample=0.4267, colsample_bytree=0.5218,
    gamma=0.2754, reg_alpha=3.2805, reg_lambda=0.6497, random_state=42
)
best_model.fit(X_train_trimmed, y_train)
print(f"Val: {best_model.score(X_val_trimmed, y_val):.1%}")

pickle.dump(best_model, open('xgb_optuna_67_7.pkl', 'wb'))
pickle.dump(X_val_trimmed.columns.tolist(), open('feature_cols_optuna.pkl', 'wb'))
fight_features.to_parquet('fight_features_optuna.parquet')
print("✔ 67.7% model locked in")

Val: 64.7%
✔ 67.7% model locked in


In [24]:
import pickle
pickle.dump(model_trimmed, open('xgb_69_2_model.pkl', 'wb'))
pickle.dump(X_val_t.columns.tolist(), open('feature_cols_69_2.pkl', 'wb'))
fight_features.to_parquet('fight_features_69_2.parquet')
print("✓ Saved")

NameError: name 'X_val_t' is not defined

In [ ]:
import pickle

model = pickle.load(open('xgb_69_model.pkl', 'rb'))
feature_cols = pickle.load(open('feature_cols_69.pkl', 'rb'))

train = fight_features[fight_features['event_date'] < '2024-01-01']
val = fight_features[fight_features['event_date'] >= '2024-01-01']

X_train_trimmed = train[feature_cols].fillna(0)
X_val_trimmed = val[feature_cols].fillna(0)
y_train = train['winner']
y_val = val['winner']

print(f"Val:   {model.score(X_val_trimmed, y_val):.1%}")
print(f"Train: {model.score(X_train_trimmed, y_train):.1%}")
print(f"Features: {len(feature_cols)}")

In [ ]:
# Load model & feature list (run once)
DB_PATH = r'C:\Users\Sarthak\Documents\ML\fighter-beta\mma_fighters.db'
conn = sqlite3.connect(DB_PATH)
# ── Fighter name → ID lookup ────────────────────────────────
fighter_name_map = pd.read_sql(
    "SELECT fighter_id, name FROM fighters_v2", conn
)
fighter_name_map['name_lower'] = fighter_name_map['name'].str.lower().str.strip()

def resolve_fighter_id(name):
    """Fuzzy-ish lookup: exact match first, then substring."""
    nl = name.lower().strip()
    exact = fighter_name_map[fighter_name_map['name_lower'] == nl]
    if len(exact) == 1:
        return exact.iloc[0]['fighter_id']
    partial = fighter_name_map[fighter_name_map['name_lower'].str.contains(nl)]
    if len(partial) == 1:
        return partial.iloc[0]['fighter_id']
    if len(partial) > 1:
        print(f"  ⚠ Ambiguous '{name}' — matches: {partial['name'].tolist()}")
    else:
        print(f"  ✗ '{name}' not found in DB")
    return None


def explain_fight(fighter_a_name, fighter_b_name, as_of_date=None,
                  fight_id=None, weight_class=None):
    """
    Full feature-by-feature breakdown using the notebook's pipelines.
    
    Args:
        fighter_a_name: Fighter 1 name (mapped to fighter_1 / f1)
        fighter_b_name: Fighter 2 name (mapped to fighter_2 / f2)
        as_of_date:     Date string 'YYYY-MM-DD' (default: today)
        fight_id:       Optional fight_id for career stats lookup
        weight_class:   Optional weight class string
    """
    # Resolve IDs
    f1_id = resolve_fighter_id(fighter_a_name)
    f2_id = resolve_fighter_id(fighter_b_name)
    if not f1_id or not f2_id:
        return None

    if as_of_date is None:
        as_of_date = datetime.now().strftime("%Y-%m-%d")

    # ── Compute all feature groups ───────────────────────────
    f1_feats = calculate_three_layer_features_v2(f1_id, f2_id, as_of_date)
    f2_feats = calculate_three_layer_features_v2(f2_id, f1_id, as_of_date)
    if f1_feats is None or f2_feats is None:
        print("  ✗ Not enough fight history for three-layer features")
        return None

    f1_feats.update(calculate_strike_features(f1_id, f2_id, as_of_date))
    f2_feats.update(calculate_strike_features(f2_id, f1_id, as_of_date))

    f1_feats.update(calculate_r1_features(f1_id, f2_id, as_of_date))
    f2_feats.update(calculate_r1_features(f2_id, f1_id, as_of_date))

    fid_placeholder = fight_id or "EXPLAIN"
    f1_feats.update(calculate_career_stats(f1_id, f2_id, fid_placeholder, as_of_date))
    f2_feats.update(calculate_career_stats(f2_id, f1_id, fid_placeholder, as_of_date))

    # ── Physical / experience features ───────────────────────
    fighters_info = pd.read_sql(f"""
        SELECT fighter_id, reach, height, dob
        FROM fighters_v2
        WHERE fighter_id IN ('{f1_id}', '{f2_id}')
    """, conn)

    phys = {}
    for fid, prefix in [(f1_id, 'f1'), (f2_id, 'f2')]:
        row = fighters_info[fighters_info['fighter_id'] == fid]
        if len(row) > 0:
            row = row.iloc[0]
            phys[f'{prefix}_reach'] = parse_reach(row['reach'])
            phys[f'{prefix}_height'] = parse_height(row['height'])
            phys[f'{prefix}_age'] = parse_age(row['dob'], as_of_date)
        else:
            phys[f'{prefix}_reach'] = None
            phys[f'{prefix}_height'] = None
            phys[f'{prefix}_age'] = None

    for fid, prefix in [(f1_id, 'f1'), (f2_id, 'f2')]:
        if fid in fighter_fights_dict:
            first = fighter_fights_dict[fid].iloc[0]['event_date']
            days = (datetime.strptime(as_of_date, "%Y-%m-%d") -
                    datetime.strptime(first, "%Y-%m-%d")).days
            phys[f'{prefix}_ufc_age'] = days / 365.25
        else:
            phys[f'{prefix}_ufc_age'] = 0

    # ── Assemble full row ────────────────────────────────────
    row = {}
    for k, v in f1_feats.items():
        row[f'f1_{k}'] = v
    for k, v in f2_feats.items():
        row[f'f2_{k}'] = v
    for k in f1_feats.keys():
        row[f'diff_{k}'] = row.get(f'f1_{k}', 0) - row.get(f'f2_{k}', 0)

    # Physical diffs
    for stat in ['age', 'reach', 'height', 'ufc_age']:
        v1 = phys.get(f'f1_{stat}')
        v2 = phys.get(f'f2_{stat}')
        if v1 is not None and v2 is not None:
            row[f'{stat}_diff'] = v1 - v2
            if v2 != 0:
                row[f'{stat}_ratio'] = v1 / v2
            else:
                row[f'{stat}_ratio'] = 1.0
        else:
            row[f'{stat}_diff'] = 0
            row[f'{stat}_ratio'] = 1.0

    row['diff_ufc_age'] = phys.get('f1_ufc_age', 0) - phys.get('f2_ufc_age', 0)

    # Interaction features
    row['age_td_interaction'] = row.get('age_diff', 0) * row.get('diff_td_avg_dec_avg', 0)
    row['age_td_adjperf_interaction'] = row.get('age_diff', 0) * row.get('diff_td_avg_adjperf', 0)
    row['age_str_interaction'] = row.get('age_diff', 0) * row.get('diff_str_acc_dec_avg', 0)
    row['age_winratio_interaction'] = row.get('age_diff', 0) * row.get('diff_win_ratio', 0)
    row['ufc_age_td_interaction'] = row.get('diff_ufc_age', 0) * row.get('diff_td_avg_dec_avg', 0)
    row['ufc_age_str_interaction'] = row.get('diff_ufc_age', 0) * row.get('diff_str_acc_dec_avg', 0)
    row['td_ctrl_interaction'] = row.get('diff_td_avg_dec_avg', 0) * row.get('diff_ctrl_time_per_min_dec_avg', 0)
    row['str_headdef_interaction'] = row.get('diff_str_acc_dec_avg', 0) * row.get('diff_head_allowed_dec_avg', 0)

    # ── Build feature vector in correct order ────────────────
    X = pd.DataFrame([{col: row.get(col, 0) for col in feature_cols}])
    X = X.fillna(0)

    # ── Predict ──────────────────────────────────────────────
    prob = model.predict_proba(X)[0][1]  # P(fighter_1 wins)
    winner = fighter_a_name if prob > 0.5 else fighter_b_name
    conf = prob if prob > 0.5 else (1 - prob)

    print(f"\n{'='*70}")
    print(f"  {fighter_a_name}  vs  {fighter_b_name}")
    print(f"  Prediction: {winner} wins  ({conf:.1%} confidence)")
    print(f"  Raw P(fighter_1): {prob:.4f}")
    print(f"{'='*70}\n")

    # ── SHAP breakdown (actual model contributions) ─────────
    import shap
    explainer_shap = shap.TreeExplainer(model)
    shap_values = explainer_shap.shap_values(X)
    base_value = explainer_shap.expected_value

    # shap_values[0] is array of per-feature contributions
    # Positive SHAP = pushes toward fighter_1 winning
    # Negative SHAP = pushes toward fighter_2 winning
    sv = shap_values[0]

    contribs = []
    for i, col in enumerate(feature_cols):
        val = X[col].iloc[0]
        shap_val = sv[i]
        contribs.append((col, val, shap_val, abs(shap_val)))

    contribs.sort(key=lambda x: x[3], reverse=True)

    print(f"Base value (avg log-odds): {base_value:.4f}")
    print(f"Sum of SHAP values: {sv.sum():+.4f}")
    print(f"{'Feature':<45} {'Value':>8} {'SHAP':>8} {'|SHAP|':>8}  Pushes toward")
    print("─" * 90)

    for name, val, shap_val, abs_shap in contribs[:25]:
        if shap_val > 0.001:
            favor = f"← {fighter_a_name[:15]}"
        elif shap_val < -0.001:
            favor = f"→ {fighter_b_name[:15]}"
        else:
            favor = "  neutral"
        print(f"{name:<45} {val:>8.3f} {shap_val:>+8.4f} {abs_shap:>8.4f}  {favor}")

    # ── Summary by category (SHAP-based) ────────────────────
    shap_dict = {col: sv[i] for i, col in enumerate(feature_cols)}

    categories = {
        'Core Stats':    [c for c in feature_cols if any(
            s in c for s in ['slpm', 'str_acc', 'td_avg', 'td_acc',
                             'sub_avg', 'ctrl_time', 'kd_per_min'])
            and 'r1_' not in c and 'interaction' not in c],
        'Strike Detail':  [c for c in feature_cols if any(
            s in c for s in ['head_', 'body_', 'leg_', 'distance_',
                             'clinch_', 'ground_', 'exchange', 'targeting',
                             'strike_share'])],
        'Round 1':        [c for c in feature_cols if 'r1_' in c],
        'Career':         [c for c in feature_cols if any(
            s in c for s in ['win_', 'ko_', 'decision_', 'sub_landing',
                             'td_defense', 'days_since', 'ctrl_ratio',
                             'sub_att_allowed', 'kd_allowed', 'td_land_ratio'])],
        'Physical':       [c for c in feature_cols if any(
            s in c for s in ['age_', 'reach_', 'height_', 'ufc_age'])
            and 'interaction' not in c],
        'Interactions':   [c for c in feature_cols if 'interaction' in c],
    }

    print(f"\n{'─'*90}")
    print(f"{'Category':<20} {'SHAP Sum':>10} {'# Feats':>8}  Pushes toward")
    print(f"{'─'*90}")

    for cat, cols in categories.items():
        cols_in = [c for c in cols if c in feature_cols]
        if not cols_in:
            continue
        shap_sum = sum(shap_dict.get(c, 0) for c in cols_in)
        direction = f"← {fighter_a_name[:12]}" if shap_sum > 0 else f"→ {fighter_b_name[:12]}"
        print(f"{cat:<20} {shap_sum:>+10.4f} {len(cols_in):>8}  {direction}")

    return prob


# ── Usage examples ───────────────────────────────────────────
# explain_fight("Kamaru Usman", "Joaquin Buckley")
# explain_fight("Islam Makhachev", "Arman Tsarukyan", as_of_date="2025-01-18")
# explain_fight("Alex Pereira", "Magomed Ankalaev")

In [25]:

def predict_fight(f1_name, f2_name, as_of_date, conn):
    """Predict a fight between two fighters."""

    # --- Look up fighter IDs ---
    # Try exact match first, then fall back to partial
    fighters_lookup = pd.read_sql("""
        SELECT fighter_id, name FROM fighters_v2
    """, conn)

    def find_fighter(name, all_fighters):
        # 1. Exact match (case-insensitive)
        exact = all_fighters[all_fighters['name'].str.lower() == name.lower()]
        if len(exact) == 1:
            return exact.iloc[0]
        # 2. Full name substring match
        full = all_fighters[all_fighters['name'].str.contains(name, case=False, na=False)]
        if len(full) == 1:
            return full.iloc[0]
        if len(full) > 1:
            print(f"  WARNING: Multiple matches for '{name}':")
            print(f"  {full[['fighter_id','name']].to_string(index=False)}")
            return full.iloc[0]
        # 3. Last name fallback
        last = name.split()[-1]
        partial = all_fighters[all_fighters['name'].str.contains(last, case=False, na=False)]
        if len(partial) > 0:
            print(f"  WARNING: No exact match for '{name}', best guess from {len(partial)} results:")
            print(f"  {partial[['fighter_id','name']].head(5).to_string(index=False)}")
            return partial.iloc[0]
        return None

    f1_match = find_fighter(f1_name, fighters_lookup)
    f2_match = find_fighter(f2_name, fighters_lookup)

    if f1_match is None or f2_match is None:
        print(f"ERROR: Could not find fighter(s). F1={f1_name}, F2={f2_name}")
        return None

    f1_id = f1_match['fighter_id']
    f2_id = f2_match['fighter_id']
    f1_db_name = f1_match['name']
    f2_db_name = f2_match['name']

    print(f"  F1: {f1_db_name} ({f1_id})")
    print(f"  F2: {f2_db_name} ({f2_id})")

    # --- Compute features ---
    f1_feats = calculate_three_layer_features_v2(f1_id, f2_id, as_of_date)
    f2_feats = calculate_three_layer_features_v2(f2_id, f1_id, as_of_date)

    if f1_feats is None or f2_feats is None:
        print("ERROR: Could not compute base features. Check fight history.")
        return None

    f1_feats.update(calculate_strike_features(f1_id, f2_id, as_of_date))
    f2_feats.update(calculate_strike_features(f2_id, f1_id, as_of_date))
    f1_feats.update(calculate_r1_features(f1_id, f2_id, as_of_date))
    f2_feats.update(calculate_r1_features(f2_id, f1_id, as_of_date))
    f1_feats.update(calculate_career_stats(f1_id, f2_id, 'PRED', as_of_date))
    f2_feats.update(calculate_career_stats(f2_id, f1_id, 'PRED', as_of_date))

    # Assemble diff features
    row = {}
    for k, v in f1_feats.items():
        row[f'f1_{k}'] = v
    for k, v in f2_feats.items():
        row[f'f2_{k}'] = v
    for k in f1_feats.keys():
        row[f'diff_{k}'] = row[f'f1_{k}'] - row[f'f2_{k}']

    # Physical features
    fighter_info = pd.read_sql(f"""
        SELECT fighter_id, reach, height, dob FROM fighters_v2
        WHERE fighter_id IN ('{f1_id}', '{f2_id}')
    """, conn)

    def get_phys(fid):
        info = fighter_info[fighter_info['fighter_id'] == fid]
        if len(info) == 0:
            return {'reach': 72, 'height': 70, 'age': 30}
        info = info.iloc[0]
        return {
            'reach': parse_reach(info['reach']) or 72,
            'height': parse_height(info['height']) or 70,
            'age': parse_age(info['dob'], as_of_date) or 30,
        }

    f1_p, f2_p = get_phys(f1_id), get_phys(f2_id)
    row['age_diff']     = f1_p['age'] - f2_p['age']
    row['age_ratio']    = f1_p['age'] / f2_p['age']
    row['reach_diff']   = f1_p['reach'] - f2_p['reach']
    row['reach_ratio']  = f1_p['reach'] / f2_p['reach']
    row['height_diff']  = f1_p['height'] - f2_p['height']
    row['height_ratio'] = f1_p['height'] / f2_p['height']

    def get_ufc_age(fid):
        if fid in fighter_fights_dict:
            first = fighter_fights_dict[fid].iloc[0]['event_date']
            return (datetime.strptime(as_of_date, "%Y-%m-%d") -
                    datetime.strptime(first, "%Y-%m-%d")).days / 365.25
        return 0
    row['diff_ufc_age'] = get_ufc_age(f1_id) - get_ufc_age(f2_id)

    # Build input
    X_pred = pd.DataFrame([row])
    for col in feature_cols:
        if col not in X_pred.columns:
            X_pred[col] = 0.0
    X_pred = X_pred[feature_cols].fillna(0)

    # Predict
    prob = model.predict_proba(X_pred)[0]
    pred = model.predict(X_pred)[0]
    winner = f1_db_name if pred == 1 else f2_db_name
    conf = max(prob) * 100

    print(f"\n  {'='*50}")
    print(f"  {f1_db_name:>25} | {prob[1]*100:5.1f}%")
    print(f"  {f2_db_name:>25} | {prob[0]*100:5.1f}%")
    print(f"  {'─'*50}")
    print(f"  Pick: {winner} ({conf:.1f}%)")
    print(f"  {'='*50}\n")

    return {'f1': f1_db_name, 'f2': f2_db_name,
            'f1_prob': prob[1], 'f2_prob': prob[0],
            'pick': winner, 'confidence': conf}


print("ANTHONY HERNANDEZ vs SEAN STRICKLAND")
predict_fight("Marlon Vera", "David Martinez", "2026-02-22", conn)



ANTHONY HERNANDEZ vs SEAN STRICKLAND
  F1: Marlon Vera (7c7332319c14094c)
  F2: David Martinez (9a97acbfd5a08bfa)


NameError: name 'feature_cols' is not defined

In [ ]:
# ============================================================
# CELL 10: Calibration & Year-by-Year Evaluation
# ============================================================
# Retrains with < 2023 cutoff so 2023, 2024, 2025 are all
# true out-of-sample. Reports accuracy, Brier score, log-loss,
# and calibration curves per year.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from xgboost import XGBClassifier
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.calibration import calibration_curve

# ── Step 1: Re-split with < 2023 cutoff ─────────────────────
train_new = fight_features[fight_features['event_date'] < '2023-01-01']
test_all  = fight_features[fight_features['event_date'] >= '2023-01-01']

test_2023 = fight_features[
    (fight_features['event_date'] >= '2023-01-01') &
    (fight_features['event_date'] <  '2024-01-01')
]
test_2024 = fight_features[
    (fight_features['event_date'] >= '2024-01-01') &
    (fight_features['event_date'] <  '2025-01-01')
]
test_2025 = fight_features[
    fight_features['event_date'] >= '2025-01-01'
]

print(f"Train (< 2023):     {len(train_new):>5} fights")
print(f"Test  2023:         {len(test_2023):>5} fights")
print(f"Test  2024:         {len(test_2024):>5} fights")
print(f"Test  2025+:        {len(test_2025):>5} fights")
print(f"Total OOS:          {len(test_all):>5} fights")

# ── Step 2: Prepare feature matrices ────────────────────────
X_train_new = train_new[feature_cols].fillna(0)
y_train_new = train_new['winner']

splits = {
    '2023':     (test_2023[feature_cols].fillna(0), test_2023['winner']),
    '2024':     (test_2024[feature_cols].fillna(0), test_2024['winner']),
    '2025+':    (test_2025[feature_cols].fillna(0), test_2025['winner']),
    'All OOS':  (test_all[feature_cols].fillna(0),  test_all['winner']),
}

# ── Step 3: Retrain with best Optuna params ──────────────────
# Using the same best params found in Cell 9
import json
best_params = json.load(open('optuna_params_69.json'))

print(f"Loaded params: {best_params}")
model_2023 = XGBClassifier(**best_params, random_state=42)
model_2023.fit(X_train_new, y_train_new)

train_acc = model_2023.score(X_train_new, y_train_new)
print(f"\nRetrained model — Train accuracy: {train_acc:.1%}")

# ── Step 4: Metrics per year ─────────────────────────────────
#
# METRIC EXPLAINER:
# ─────────────────
# Accuracy:    % of fights where the model picked the right winner.
#              Easy to understand but doesn't reward confident correct
#              picks or penalize confident wrong picks.
#
# Brier Score: Measures how accurate the *probabilities* are, not just
#              the picks. Range 0→1, lower is better.
#              Think of it as "average squared error of your probability."
#              A model that always says 50/50 scores ~0.25.
#              A perfect model scores 0.0. Anything < 0.23 is decent.
#
# Log-Loss:    Also measures probability accuracy but punishes confident
#              wrong predictions much harder than Brier score does.
#              Lower is better. A 50/50 model scores ~0.693.
#              Good MMA models typically land around 0.60–0.65.
#              Think of it as: "how surprised was the model when it was wrong?"

print(f"\n{'='*70}")
print(f"{'YEAR-BY-YEAR EVALUATION (all OOS — trained on < 2023)'}")
print(f"{'='*70}")
print(f"{'Year':<10} {'N':>5} {'Accuracy':>10} {'Brier↓':>10} {'LogLoss↓':>10}  {'Interpretation'}")
print(f"{'─'*70}")

results = {}
for year, (X, y) in splits.items():
    if len(X) == 0:
        continue
    probs = model_2023.predict_proba(X)[:, 1]
    acc   = model_2023.score(X, y)
    brier = brier_score_loss(y, probs)
    ll    = log_loss(y, probs)

    # Simple interpretation flags
    if acc >= 0.68:
        interp = "✓ Strong"
    elif acc >= 0.64:
        interp = "✓ Good"
    elif acc >= 0.60:
        interp = "~ Okay"
    else:
        interp = "✗ Weak"

    results[year] = {'acc': acc, 'brier': brier, 'logloss': ll,
                     'probs': probs, 'y': y, 'n': len(y)}

    print(f"{year:<10} {len(y):>5} {acc:>9.1%} {brier:>10.4f} {ll:>10.4f}  {interp}")

print(f"{'─'*70}")
print(f"Reference — a 50/50 model would score: acc=50.0%  brier=0.2500  logloss=0.6931")

# ── Step 5: Calibration curves ───────────────────────────────
#
# CALIBRATION CURVE EXPLAINER:
# A perfectly calibrated model: when it says "70% chance fighter 1 wins",
# fighter 1 actually wins 70% of the time.
# The diagonal line = perfect calibration.
# Above diagonal = model is UNDERCONFIDENT (says 60%, reality is 70%)
# Below diagonal = model is OVERCONFIDENT  (says 70%, reality is 60%)

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

colors = {'2023': '#2196F3', '2024': '#4CAF50', '2025+': '#FF9800', 'All OOS': '#9C27B0'}

year_positions = {'2023': (0,0), '2024': (0,1), '2025+': (0,2), 'All OOS': (1,0)}

for year, pos in year_positions.items():
    if year not in results:
        continue
    r   = results[year]
    ax  = fig.add_subplot(gs[pos[0], pos[1]])

    # Calibration curve
    try:
        n_bins = min(10, max(5, r['n'] // 30))
        frac_pos, mean_pred = calibration_curve(r['y'], r['probs'],
                                                 n_bins=n_bins, strategy='quantile')
        ax.plot(mean_pred, frac_pos, 'o-', color=colors[year],
                linewidth=2, markersize=6, label='Model')
    except Exception as e:
        ax.text(0.5, 0.5, f'Not enough data\n({r["n"]} fights)',
                ha='center', va='center', transform=ax.transAxes)

    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect', alpha=0.6)
    ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Predicted probability', fontsize=10)
    ax.set_ylabel('Actual win rate', fontsize=10)
    ax.set_title(
        f"{year}  |  Acc: {r['acc']:.1%}  Brier: {r['brier']:.4f}  LL: {r['logloss']:.4f}",
        fontsize=10, fontweight='bold'
    )
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Annotate under/overconfidence
    if len(frac_pos) > 0:
        mid = len(frac_pos) // 2
        diff = frac_pos[mid] - mean_pred[mid]
        if diff > 0.05:
            ax.text(0.05, 0.88, '↑ Underconfident', transform=ax.transAxes,
                    color='blue', fontsize=8)
        elif diff < -0.05:
            ax.text(0.05, 0.88, '↓ Overconfident', transform=ax.transAxes,
                    color='red', fontsize=8)
        else:
            ax.text(0.05, 0.88, '✓ Well calibrated', transform=ax.transAxes,
                    color='green', fontsize=8)

# ── Step 6: Confidence distribution per year ────────────────
# Shows: are most predictions close to 50/50 (uncertain)
# or are there lots of high-confidence picks?
ax_dist = fig.add_subplot(gs[1, 1:])

for year, r in results.items():
    if year == 'All OOS':
        continue
    ax_dist.hist(r['probs'], bins=20, alpha=0.5, color=colors[year],
                 label=year, density=True)

ax_dist.axvline(0.5, color='black', linestyle='--', linewidth=1, alpha=0.7)
ax_dist.set_xlabel('Predicted P(Fighter 1 wins)', fontsize=10)
ax_dist.set_ylabel('Density', fontsize=10)
ax_dist.set_title('Confidence Distribution by Year\n'
                  '(clustered near 0.5 = uncertain model, spread out = confident)',
                  fontsize=10, fontweight='bold')
ax_dist.legend(fontsize=9)
ax_dist.grid(True, alpha=0.3)

fig.suptitle('MMA Model — Calibration & Year-by-Year Evaluation\n'
             'Trained on fights before 2023 | All years shown are true OOS',
             fontsize=13, fontweight='bold', y=1.01)

plt.savefig('calibration_eval.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✔ Saved calibration_eval.png")

# ── Step 7: Confidence accuracy breakdown ───────────────────
# Answers: "when the model is more confident, is it actually more accurate?"
print(f"\n{'='*70}")
print("CONFIDENCE vs ACCURACY (All OOS)")
print("Answers: do higher-confidence picks win more often?")
print(f"{'─'*70}")
print(f"{'Confidence Band':<22} {'N Fights':>9} {'Accuracy':>10} {'Avg Confidence':>16}")
print(f"{'─'*70}")

all_probs = results['All OOS']['probs']
all_y     = results['All OOS']['y'].values
conf      = np.maximum(all_probs, 1 - all_probs)  # confidence = distance from 50%
preds     = (all_probs >= 0.5).astype(int)
correct   = (preds == all_y).astype(int)

bands = [(0.50, 0.55), (0.55, 0.60), (0.60, 0.65), (0.65, 0.70), (0.70, 1.01)]
for lo, hi in bands:
    mask = (conf >= lo) & (conf < hi)
    n    = mask.sum()
    if n == 0:
        continue
    band_acc  = correct[mask].mean()
    band_conf = conf[mask].mean()
    label = f"{lo*100:.0f}%–{min(hi,1)*100:.0f}%"
    bar   = '█' * int(band_acc * 20)
    print(f"  {label:<20} {n:>9} {band_acc:>9.1%}   {band_conf:>13.1%}  {bar}")

print(f"\n✔ Done. Key question: does accuracy rise as confidence rises?")
print(f"   If yes → model is well-calibrated and trustworthy at high confidence.")
print(f"   If flat → model's confidence scores don't mean much.")

In [ ]:
# ============================================================
# CELL 11: Favorite vs Underdog Calibration
# ============================================================
# A "favorite" pick = model confidence >= 55% on either fighter
# An "underdog" pick = model picked the fighter with < 50% implied
#   probability (i.e. the model disagreed with its own confidence
#   threshold, or we define underdog as conf < 60%)
#
# More practically:
#   Favorite  = model confidence >= 60%  (model has a clear lean)
#   Toss-up   = model confidence 50–60%  (model is uncertain)
#   Upset pick = model picked correctly but was < 55% confident
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss

# ── Build combined OOS dataframe ─────────────────────────────
test_all = fight_features[fight_features['event_date'] >= '2023-01-01'].copy()
X_all    = test_all[feature_cols].fillna(0)
y_all    = test_all['winner'].values

probs    = model_2023.predict_proba(X_all)[:, 1]
preds    = (probs >= 0.5).astype(int)
correct  = (preds == y_all).astype(int)
conf     = np.maximum(probs, 1 - probs)  # distance from 50%

# Attach to dataframe for slicing
test_all = test_all.copy()
test_all['prob_f1']  = probs
test_all['conf']     = conf
test_all['pred']     = preds
test_all['correct']  = correct
test_all['year']     = test_all['event_date'].str[:4]

# ── Define segments ──────────────────────────────────────────
# Favorite:  model >= 60% confident
# Toss-up:   model 50–60% confident
# These are model-defined, not odds-defined
fav_mask    = test_all['conf'] >= 0.60
tossup_mask = test_all['conf'] <  0.60

segments = {
    'Favorite (conf ≥ 60%)': test_all[fav_mask],
    'Toss-up  (conf < 60%)': test_all[tossup_mask],
}

# ── Print metrics per segment and year ───────────────────────
print(f"{'='*75}")
print(f"FAVORITE vs TOSS-UP BREAKDOWN")
print(f"{'='*75}")

for seg_name, seg_df in segments.items():
    print(f"\n── {seg_name} ──")
    print(f"  {'Year':<10} {'N':>6} {'Accuracy':>10} {'Brier↓':>10} {'LogLoss↓':>10}  {'Avg Conf':>10}")
    print(f"  {'─'*65}")

    for year in ['2023', '2024', '2025']:
        ydf = seg_df[seg_df['year'] == year]
        if len(ydf) < 10:
            print(f"  {year:<10} {len(ydf):>6}  (too few samples)")
            continue
        p  = ydf['prob_f1'].values
        y  = ydf['winner'].values
        ac = ydf['correct'].mean()
        br = brier_score_loss(y, p)
        ll = log_loss(y, p)
        cf = ydf['conf'].mean()
        print(f"  {year:<10} {len(ydf):>6} {ac:>9.1%} {br:>10.4f} {ll:>10.4f} {cf:>10.1%}")

    # All OOS
    p  = seg_df['prob_f1'].values
    y  = seg_df['winner'].values
    if len(seg_df) >= 10:
        ac = seg_df['correct'].mean()
        br = brier_score_loss(y, p)
        ll = log_loss(y, p)
        cf = seg_df['conf'].mean()
        print(f"  {'─'*65}")
        print(f"  {'All OOS':<10} {len(seg_df):>6} {ac:>9.1%} {br:>10.4f} {ll:>10.4f} {cf:>10.1%}")

# ── Confidence band breakdown ─────────────────────────────────
print(f"\n{'='*75}")
print(f"ACCURACY BY CONFIDENCE BAND (All OOS)")
print(f"Answers: when the model is more sure, does it win more?")
print(f"{'─'*75}")
print(f"  {'Band':<18} {'N':>6} {'Accuracy':>10} {'Avg Conf':>10}  Bar")
print(f"  {'─'*65}")

bands = [(0.50, 0.55), (0.55, 0.60), (0.60, 0.65), (0.65, 0.70), (0.70, 0.75), (0.75, 1.01)]
band_accs, band_labels, band_ns = [], [], []

for lo, hi in bands:
    mask  = (test_all['conf'] >= lo) & (test_all['conf'] < hi)
    n     = mask.sum()
    if n == 0:
        continue
    ac    = test_all.loc[mask, 'correct'].mean()
    cf    = test_all.loc[mask, 'conf'].mean()
    label = f"{lo*100:.0f}–{min(hi,1.0)*100:.0f}%"
    bar   = '█' * int(ac * 30)
    print(f"  {label:<18} {n:>6} {ac:>9.1%} {cf:>10.1%}  {bar}")
    band_accs.append(ac)
    band_labels.append(label)
    band_ns.append(n)

# ── Plots ─────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 11))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

years       = ['2023', '2024', '2025']
year_colors = {'2023': '#2196F3', '2024': '#4CAF50', '2025': '#FF9800'}

# Row 1: Calibration curves split by favorite / toss-up
for col_idx, (seg_name, seg_df) in enumerate(segments.items()):
    ax = fig.add_subplot(gs[0, col_idx])
    for year in years:
        ydf = seg_df[seg_df['year'] == year]
        if len(ydf) < 20:
            continue
        try:
            n_bins = min(8, max(4, len(ydf) // 25))
            fp, mp = calibration_curve(
                ydf['winner'].values,
                ydf['prob_f1'].values,
                n_bins=n_bins, strategy='quantile'
            )
            ax.plot(mp, fp, 'o-', color=year_colors[year],
                    linewidth=2, markersize=5, label=f'{year} (n={len(ydf)})')
        except Exception:
            pass

    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Perfect')
    ax.set_xlim(0.3, 0.9)
    ax.set_ylim(0.2, 1.0)
    ax.set_xlabel('Predicted probability', fontsize=9)
    ax.set_ylabel('Actual win rate', fontsize=9)
    ax.set_title(seg_name, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Row 1 col 3: Accuracy by confidence band (bar chart)
ax_band = fig.add_subplot(gs[0, 2])
colors_band = ['#ef5350' if a < 0.60 else '#66bb6a' if a >= 0.65 else '#ffa726'
               for a in band_accs]
bars = ax_band.bar(band_labels, band_accs, color=colors_band, edgecolor='white', linewidth=0.8)
ax_band.axhline(0.5,  color='red',   linestyle='--', linewidth=1, alpha=0.6, label='Coin flip')
ax_band.axhline(0.645, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='Overall avg')

for bar, ac, n in zip(bars, band_accs, band_ns):
    ax_band.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{ac:.1%}\n(n={n})', ha='center', va='bottom', fontsize=8)

ax_band.set_ylim(0, 1.0)
ax_band.set_xlabel('Confidence band', fontsize=9)
ax_band.set_ylabel('Accuracy', fontsize=9)
ax_band.set_title('Accuracy by Confidence Band\n(All OOS)', fontsize=10, fontweight='bold')
ax_band.legend(fontsize=8)
ax_band.grid(True, alpha=0.3, axis='y')
plt.setp(ax_band.xaxis.get_majorticklabels(), rotation=30, ha='right')

# Row 2: Accuracy by year × segment (grouped bar)
ax_grp = fig.add_subplot(gs[1, :2])
x      = np.arange(len(years))
width  = 0.35

fav_accs    = []
tossup_accs = []
fav_ns      = []
tossup_ns   = []

for year in years:
    fdf = segments['Favorite (conf ≥ 60%)']
    tdf = segments['Toss-up  (conf < 60%)']
    fy  = fdf[fdf['year'] == year]
    ty  = tdf[tdf['year'] == year]
    fav_accs.append(fy['correct'].mean() if len(fy) >= 5 else 0)
    tossup_accs.append(ty['correct'].mean() if len(ty) >= 5 else 0)
    fav_ns.append(len(fy))
    tossup_ns.append(len(ty))

b1 = ax_grp.bar(x - width/2, fav_accs,    width, label='Favorite (≥60%)',
                color='#42a5f5', edgecolor='white')
b2 = ax_grp.bar(x + width/2, tossup_accs, width, label='Toss-up (<60%)',
                color='#ffa726', edgecolor='white')

ax_grp.axhline(0.5, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Coin flip')
ax_grp.set_xticks(x)
ax_grp.set_xticklabels(years, fontsize=11)
ax_grp.set_ylabel('Accuracy', fontsize=10)
ax_grp.set_ylim(0, 1.0)
ax_grp.set_title('Favorite vs Toss-up Accuracy by Year',
                 fontsize=11, fontweight='bold')
ax_grp.legend(fontsize=9)
ax_grp.grid(True, alpha=0.3, axis='y')

for bar, ac, n in zip(b1, fav_accs, fav_ns):
    ax_grp.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{ac:.1%}\n(n={n})', ha='center', va='bottom', fontsize=8)
for bar, ac, n in zip(b2, tossup_accs, tossup_ns):
    ax_grp.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{ac:.1%}\n(n={n})', ha='center', va='bottom', fontsize=8)

# Row 2 col 3: % of predictions that are favorites vs toss-ups per year
ax_pie = fig.add_subplot(gs[1, 2])
split_data = [[fav_ns[i], tossup_ns[i]] for i in range(len(years))]
bottom = np.zeros(len(years))
colors_split = ['#42a5f5', '#ffa726']
labels_split = ['Favorite (≥60%)', 'Toss-up (<60%)']

totals = [fav_ns[i] + tossup_ns[i] for i in range(len(years))]
for j, (color, label) in enumerate(zip(colors_split, labels_split)):
    vals = [split_data[i][j] / totals[i] * 100 for i in range(len(years))]
    ax_pie.bar(years, vals, bottom=bottom, color=color,
               label=label, edgecolor='white')
    for i, (v, b) in enumerate(zip(vals, bottom)):
        ax_pie.text(i, b + v/2, f'{v:.0f}%', ha='center',
                    va='center', fontsize=9, color='white', fontweight='bold')
    bottom += np.array(vals)

ax_pie.set_ylabel('% of predictions', fontsize=10)
ax_pie.set_title('Prediction Mix by Year\n(how often model is confident)',
                 fontsize=10, fontweight='bold')
ax_pie.legend(fontsize=8)
ax_pie.set_ylim(0, 100)
ax_pie.grid(True, alpha=0.2, axis='y')

fig.suptitle('MMA Model — Favorite vs Toss-up Calibration\nAll OOS (trained < 2023)',
             fontsize=13, fontweight='bold', y=1.01)

plt.savefig('fav_underdog_calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✔ Saved fav_underdog_calibration.png")

In [ ]:
# ============================================================
# CELL 12: Model vs Odds Calibration Analysis
# ============================================================
# Compares model predictions against closing betting odds.
# Key questions:
#   1. When model agrees with odds favorite — how accurate?
#   2. When model disagrees (picks the underdog) — how accurate?
#   3. Does the model add value over just following the odds?
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss

# ── Step 1: Load and merge odds ──────────────────────────────
odds = pd.read_sql("""
    SELECT 
        fo.fight_id,
        fo.fighter_1_close_low,
        fo.fighter_2_close_low
    FROM fight_odds fo
    WHERE fo.fighter_1_close_low IS NOT NULL
      AND fo.fighter_2_close_low IS NOT NULL
""", conn)

# ── Step 2: Convert American odds to implied probability ─────
# American odds → implied prob:
#   Negative odds (favorite): prob = |odds| / (|odds| + 100)
#   Positive odds (underdog): prob = 100 / (odds + 100)
# Then normalize both to remove the vig (overround)

def american_to_prob(odds_val):
    if odds_val < 0:
        return abs(odds_val) / (abs(odds_val) + 100)
    else:
        return 100 / (odds_val + 100)

odds['raw_p_f1'] = odds['fighter_1_close_low'].apply(american_to_prob)
odds['raw_p_f2'] = odds['fighter_2_close_low'].apply(american_to_prob)

# Remove vig — normalize so they sum to 1
total = odds['raw_p_f1'] + odds['raw_p_f2']
odds['odds_prob_f1'] = odds['raw_p_f1'] / total  # true implied prob f1 wins
odds['odds_prob_f2'] = odds['raw_p_f2'] / total

# Odds favorite = fighter with higher implied probability
odds['odds_fav_is_f1'] = odds['odds_prob_f1'] >= odds['odds_prob_f2']

# ── Step 3: Merge with OOS fight features ────────────────────
test_all = fight_features[fight_features['event_date'] >= '2023-01-01'].copy()
X_all    = test_all[feature_cols].fillna(0)
y_all    = test_all['winner'].values

model_probs = model_2023.predict_proba(X_all)[:, 1]
test_all['model_prob_f1'] = model_probs
test_all['model_pred_f1'] = (model_probs >= 0.5).astype(int)
test_all['model_conf']    = np.maximum(model_probs, 1 - model_probs)
test_all['year']          = test_all['event_date'].str[:4]

# Merge odds
df = test_all.merge(odds, on='fight_id', how='inner')
print(f"✔ Fights with both model predictions and odds: {len(df)}")
print(f"  Year breakdown:")
print(df.groupby('year').size().to_string())

# ── Step 4: Define agreement/disagreement ────────────────────
# Model pick: f1 if model_prob_f1 >= 0.5
# Odds pick:  f1 if odds_prob_f1 >= 0.5
df['model_picks_f1'] = (df['model_prob_f1'] >= 0.5).astype(int)
df['odds_picks_f1']  = (df['odds_prob_f1']  >= 0.5).astype(int)

df['agree']    = df['model_picks_f1'] == df['odds_picks_f1']
df['disagree'] = ~df['agree']

# Correctness
df['model_correct'] = (df['model_picks_f1'] == df['winner']).astype(int)
df['odds_correct']  = (df['odds_picks_f1']  == df['winner']).astype(int)

# When they disagree, did the model or odds win?
disagree_df = df[df['disagree']].copy()

# ── Step 5: Text summary ─────────────────────────────────────
print(f"\n{'='*70}")
print(f"MODEL vs ODDS — AGREEMENT ANALYSIS (All OOS 2023–2025)")
print(f"{'='*70}")

# Overall
print(f"\n── Overall ──")
print(f"  Model accuracy (all):     {df['model_correct'].mean():.1%}  (n={len(df)})")
print(f"  Odds accuracy (all):      {df['odds_correct'].mean():.1%}  (n={len(df)})")
print(f"  Fights they agree:        {df['agree'].sum()} ({df['agree'].mean():.1%})")
print(f"  Fights they disagree:     {df['disagree'].sum()} ({df['disagree'].mean():.1%})")

# When they agree
agree_df = df[df['agree']]
print(f"\n── When Model & Odds AGREE ──")
print(f"  Accuracy:   {agree_df['model_correct'].mean():.1%}  (n={len(agree_df)})")
print(f"  Avg model conf: {agree_df['model_conf'].mean():.1%}")

# When they disagree
print(f"\n── When Model & Odds DISAGREE (model picks underdog) ──")
print(f"  Model accuracy: {disagree_df['model_correct'].mean():.1%}  (n={len(disagree_df)})")
print(f"  Odds accuracy:  {disagree_df['odds_correct'].mean():.1%}  (n={len(disagree_df)})")
print(f"  Avg model conf: {disagree_df['model_conf'].mean():.1%}")
print(f"  → Model beat odds: {(disagree_df['model_correct'].mean() > disagree_df['odds_correct'].mean())}")

# By year
print(f"\n── By Year ──")
print(f"  {'Year':<8} {'N':>5} {'Agree%':>8} {'Model Acc':>11} {'Odds Acc':>10} {'Disagree N':>12} {'Model on Disagree':>18} {'Odds on Disagree':>17}")
print(f"  {'─'*90}")
for year in ['2023', '2024', '2025']:
    ydf  = df[df['year'] == year]
    ydis = ydf[ydf['disagree']]
    if len(ydf) == 0:
        continue
    print(f"  {year:<8} {len(ydf):>5} "
          f"{ydf['agree'].mean():>7.1%} "
          f"{ydf['model_correct'].mean():>10.1%} "
          f"{ydf['odds_correct'].mean():>9.1%} "
          f"{len(ydis):>12} "
          f"{ydis['model_correct'].mean() if len(ydis)>0 else 0:>17.1%} "
          f"{ydis['odds_correct'].mean() if len(ydis)>0 else 0:>16.1%}")

# Model confidence on disagreements
print(f"\n── Model Confidence on Disagreements (when it picks the underdog) ──")
print(f"  {'Conf Band':<18} {'N':>5} {'Model Acc':>11} {'Odds Acc':>10}")
print(f"  {'─'*50}")
for lo, hi in [(0.50, 0.55), (0.55, 0.60), (0.60, 0.65), (0.65, 1.01)]:
    mask = (disagree_df['model_conf'] >= lo) & (disagree_df['model_conf'] < hi)
    n = mask.sum()
    if n < 5:
        continue
    m_acc = disagree_df.loc[mask, 'model_correct'].mean()
    o_acc = disagree_df.loc[mask, 'odds_correct'].mean()
    label = f"{lo*100:.0f}–{min(hi,1)*100:.0f}%"
    flag  = "✓ Model wins" if m_acc > o_acc else "✗ Odds win"
    print(f"  {label:<18} {n:>5} {m_acc:>10.1%} {o_acc:>9.1%}  {flag}")

# ── Step 6: Plots ─────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
year_colors = {'2023': '#2196F3', '2024': '#4CAF50', '2025': '#FF9800'}

# ── Plot 1: Calibration — Model vs Odds (All OOS) ────────────
ax1 = fig.add_subplot(gs[0, 0])
# Model calibration
try:
    fp_m, mp_m = calibration_curve(df['winner'], df['model_prob_f1'],
                                    n_bins=10, strategy='quantile')
    ax1.plot(mp_m, fp_m, 'o-', color='#2196F3', linewidth=2,
             markersize=6, label='Model')
except Exception:
    pass
# Odds calibration
try:
    fp_o, mp_o = calibration_curve(df['winner'], df['odds_prob_f1'],
                                    n_bins=10, strategy='quantile')
    ax1.plot(mp_o, fp_o, 's-', color='#f44336', linewidth=2,
             markersize=6, label='Odds')
except Exception:
    pass
ax1.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Perfect')
ax1.set_xlim(0.2, 0.9)
ax1.set_ylim(0.2, 0.9)
ax1.set_xlabel('Predicted probability', fontsize=9)
ax1.set_ylabel('Actual win rate', fontsize=9)
ax1.set_title('Model vs Odds Calibration\n(All OOS)', fontsize=10, fontweight='bold')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# ── Plot 2: Accuracy — Agree vs Disagree by year ─────────────
ax2 = fig.add_subplot(gs[0, 1])
years = ['2023', '2024', '2025']
x = np.arange(len(years))
w = 0.25

agree_accs    = [df[(df['year']==y) & df['agree']]['model_correct'].mean() for y in years]
dis_model     = [df[(df['year']==y) & df['disagree']]['model_correct'].mean() for y in years]
dis_odds      = [df[(df['year']==y) & df['disagree']]['odds_correct'].mean() for y in years]
agree_ns      = [df[(df['year']==y) & df['agree']].shape[0] for y in years]
dis_ns        = [df[(df['year']==y) & df['disagree']].shape[0] for y in years]

b1 = ax2.bar(x - w, agree_accs, w, label='Agree (model=odds)',
             color='#66bb6a', edgecolor='white')
b2 = ax2.bar(x,     dis_model,  w, label='Disagree — Model',
             color='#2196F3', edgecolor='white')
b3 = ax2.bar(x + w, dis_odds,   w, label='Disagree — Odds',
             color='#f44336', edgecolor='white')

ax2.axhline(0.5, color='red', linestyle='--', linewidth=1, alpha=0.5)
ax2.set_xticks(x)
ax2.set_xticklabels(years)
ax2.set_ylim(0, 1.0)
ax2.set_ylabel('Accuracy', fontsize=9)
ax2.set_title('Agreement vs Disagreement\nAccuracy by Year', fontsize=10, fontweight='bold')
ax2.legend(fontsize=7)
ax2.grid(True, alpha=0.3, axis='y')

for bar, ac, n in zip(b1, agree_accs, agree_ns):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f'{ac:.0%}\n({n})', ha='center', va='bottom', fontsize=7)
for bar, ac, n in zip(b2, dis_model, dis_ns):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f'{ac:.0%}\n({n})', ha='center', va='bottom', fontsize=7)
for bar, ac in zip(b3, dis_odds):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f'{ac:.0%}', ha='center', va='bottom', fontsize=7)

# ── Plot 3: Model confidence distribution on disagreements ────
ax3 = fig.add_subplot(gs[0, 2])
for year in years:
    ydf = df[(df['year'] == year) & df['disagree']]
    if len(ydf) > 0:
        ax3.hist(ydf['model_conf'], bins=12, alpha=0.5,
                 color=year_colors[year], label=f'{year} (n={len(ydf)})',
                 density=True)
ax3.axvline(0.6, color='black', linestyle='--', linewidth=1.5,
            label='60% threshold', alpha=0.7)
ax3.set_xlabel('Model confidence', fontsize=9)
ax3.set_ylabel('Density', fontsize=9)
ax3.set_title('Model Confidence\nWhen Picking the Underdog', fontsize=10, fontweight='bold')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)

# ── Plot 4: Calibration by year — model only ─────────────────
ax4 = fig.add_subplot(gs[1, 0])
for year in years:
    ydf = df[df['year'] == year]
    if len(ydf) < 20:
        continue
    try:
        fp, mp = calibration_curve(ydf['winner'], ydf['model_prob_f1'],
                                    n_bins=8, strategy='quantile')
        ax4.plot(mp, fp, 'o-', color=year_colors[year], linewidth=2,
                 markersize=5, label=f'{year} (n={len(ydf)})')
    except Exception:
        pass
ax4.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
ax4.set_xlim(0.3, 0.85)
ax4.set_ylim(0.2, 0.95)
ax4.set_xlabel('Predicted probability', fontsize=9)
ax4.set_ylabel('Actual win rate', fontsize=9)
ax4.set_title('Model Calibration by Year\n(Odds-matched fights only)', fontsize=10, fontweight='bold')
ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3)

# ── Plot 5: Odds implied prob vs model prob scatter ───────────
ax5 = fig.add_subplot(gs[1, 1])
agree_pts   = df[df['agree']]
disagree_pts = df[df['disagree']]

ax5.scatter(agree_pts['odds_prob_f1'],    agree_pts['model_prob_f1'],
            alpha=0.3, s=12, color='#66bb6a', label='Agree')
ax5.scatter(disagree_pts['odds_prob_f1'], disagree_pts['model_prob_f1'],
            alpha=0.5, s=15, color='#f44336', label='Disagree')
ax5.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.4)
ax5.axhline(0.5, color='gray', linewidth=0.8, alpha=0.4)
ax5.axvline(0.5, color='gray', linewidth=0.8, alpha=0.4)
ax5.set_xlabel('Odds implied probability (F1)', fontsize=9)
ax5.set_ylabel('Model probability (F1)', fontsize=9)
ax5.set_title('Model vs Odds Probability\n(Red = model disagrees with odds)',
              fontsize=10, fontweight='bold')
ax5.legend(fontsize=8)
ax5.set_xlim(0.1, 0.9)
ax5.set_ylim(0.1, 0.9)
ax5.grid(True, alpha=0.3)

# ── Plot 6: Model edge — prob diff when disagreeing ───────────
ax6 = fig.add_subplot(gs[1, 2])
# Edge = how far the model's prob is from the odds prob
disagree_df = df[df['disagree']].copy()
disagree_df['edge'] = abs(disagree_df['model_prob_f1'] - disagree_df['odds_prob_f1'])

# Bin by edge size and show accuracy
edge_bins   = [0, 0.05, 0.10, 0.15, 0.20, 1.0]
edge_labels = ['0–5%', '5–10%', '10–15%', '15–20%', '20%+']
edge_accs, edge_ns = [], []

for lo, hi in zip(edge_bins[:-1], edge_bins[1:]):
    mask = (disagree_df['edge'] >= lo) & (disagree_df['edge'] < hi)
    n    = mask.sum()
    ac   = disagree_df.loc[mask, 'model_correct'].mean() if n > 0 else 0
    edge_accs.append(ac)
    edge_ns.append(n)

colors_edge = ['#ef5350' if a < 0.5 else '#66bb6a' if a >= 0.55 else '#ffa726'
               for a in edge_accs]
bars = ax6.bar(edge_labels, edge_accs, color=colors_edge, edgecolor='white')
ax6.axhline(0.5, color='red', linestyle='--', linewidth=1, alpha=0.6, label='Coin flip')

for bar, ac, n in zip(bars, edge_accs, edge_ns):
    ax6.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f'{ac:.1%}\n(n={n})', ha='center', va='bottom', fontsize=8)

ax6.set_ylim(0, 1.0)
ax6.set_xlabel('Model vs Odds probability gap', fontsize=9)
ax6.set_ylabel('Model accuracy', fontsize=9)
ax6.set_title('Model Accuracy by Size of Disagreement\n(bigger gap = model is more different from odds)',
              fontsize=10, fontweight='bold')
ax6.legend(fontsize=8)
ax6.grid(True, alpha=0.3, axis='y')

fig.suptitle('MMA Model vs Betting Odds — Calibration & Disagreement Analysis\n'
             'All OOS (2023–2025) | Closing odds (bestfightodds)',
             fontsize=13, fontweight='bold', y=1.01)

plt.savefig('model_vs_odds_calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✔ Saved model_vs_odds_calibration.png")